# Generative AI & NLP Evaluation: Source-Grounded Historical Text Analysis
## An Executed GPT Evaluation Case Study

**Author:** Roxanne Lyle  
**Source text:** Abraham Lincoln's *Gettysburg Address*  
**Authoritative source:** National Park Service  
**Executed model:** `gpt-5-mini`

---

## Case Study Purpose

This case study evaluates how a pretrained GPT-like language model interprets, explains, and summarizes a historically important source text. The National Park Service version of the *Gettysburg Address* serves as the authoritative evidence base for evaluating generated responses.

The analysis examines three central NLP capabilities:

1. Source-grounded interpretation of historical passages.
2. Explanation of formal nineteenth-century language and rhetorical ideas.
3. Constrained summarization under different prompt designs.

Rather than treating fluent output as evidence of factual understanding, responses are evaluated for historical fidelity, contextual relevance, theme coverage, clarity, instruction compliance, and unsupported content. Findings are limited to the selected model, source text, prompts, and executed trials.

## Case Study Roadmap

1. Establish the authoritative source and theoretical foundation.
2. Configure a secure and reproducible API workflow.
3. Generate and evaluate passage interpretations and formal-language explanations.
4. Compare source-free and source-grounded summarization strategies.
5. Measure repeated-trial variability and exact constraint compliance.
6. Screen external-context claims and quotation fidelity.
7. Conduct structured human evaluation across six quality criteria.
8. Consolidate strengths, limitations, applications, and safeguards.


## Evaluation Framework

The model’s responses are evaluated against the National Park Service version of the *Gettysburg Address*, which serves as the authoritative source text for this notebook. The analysis does not assume that a fluent or persuasive response is historically accurate. Instead, each major output is reviewed using multiple criteria that address both response quality and evidence boundaries.

| Evaluation criterion | Focus of the evaluation |
|---|---|
| **Historical fidelity** | Whether the response accurately represents the ideas and language contained in the speech. |
| **Contextual relevance** | Whether the response directly and fully addresses the prompt. |
| **Theme coverage** | Whether central ideas such as equality, sacrifice, freedom, national purpose, responsibility of the living, and democratic government are represented appropriately. |
| **Formal-language interpretation** | Whether nineteenth-century wording, metaphors, repetition, and rhetorical structure are explained without changing their meaning. |
| **Coherence and clarity** | Whether the response is logically organized, understandable, and appropriate for the task. |
| **Instruction compliance** | Whether the response follows the requested scope, format, length, and source restrictions. |
| **Unsupported content** | Whether the response introduces claims, quotations, dates, or historical details that are not supported by the supplied speech. |
| **Consistency across trials** | Whether repeated responses preserve the same central meaning even when wording or emphasis varies. |

Quantitative checks, including word counts, lexical theme detection, and response-similarity measures, are used only as supporting evidence. These measures cannot independently determine historical accuracy, contextual understanding, or interpretive quality. Each output therefore receives a written human interpretation based on direct comparison with the source text.

The prompt-comparison and repeated-trial experiments are descriptive rather than causal. Differences among outputs may be associated with prompt wording or normal model variability, but the limited number of trials cannot establish that one prompt design caused a particular result. Broader historical claims that extend beyond the speech are identified for external verification rather than automatically treated as errors or confirmed facts.

# 1. Historical Context and Theoretical Foundation

## Historical Context and Authoritative Source

Abraham Lincoln delivered the *Gettysburg Address* on November 19, 1863, during the dedication of the Soldiers’ National Cemetery in Gettysburg, Pennsylvania. Although the speech is brief, it connects the sacrifice of Civil War soldiers with broader principles of equality, national endurance, freedom, civic responsibility, and democratic government. The National Park Service version of the speech serves as the authoritative source text for this notebook.

Establishing an authoritative source is essential because a language model may generate historically plausible statements that are not directly supported by the document being analyzed. The model’s responses are therefore evaluated first against the language and ideas contained in the speech. Broader claims about Lincoln’s intentions, the Civil War, slavery, military events, or nineteenth-century society are treated as contextual interpretations that require verification through additional authoritative historical sources.

**Authoritative source:** National Park Service. (2015, April 10). *Gettysburg Address*. https://www.nps.gov/liho/learn/historyculture/gettysburgaddress.htm



## Transformer Architecture, Generative Pretraining, and Historical Text Analysis

GPT-like language models are based on the transformer architecture introduced by Vaswani et al. (2017). One of the transformer’s central components is self-attention, which allows the model to represent relationships among words and phrases across an entire sequence rather than interpreting each term only through its immediate surroundings. This capability is especially relevant to the *Gettysburg Address* because the meaning of expressions such as “unfinished work,” “new birth of freedom,” and “government of the people” depends on ideas developed across multiple parts of the speech. Attention mechanisms may help the model connect these expressions with broader themes of equality, sacrifice, national endurance, freedom, and civic responsibility.

GPT-like models also rely on generative pretraining. During pretraining, a model learns statistical patterns in vocabulary, grammar, sentence structure, style, and discourse from large collections of text. Radford et al. (2018) demonstrated that generative pretraining can support a range of language tasks because the model develops broad language representations before receiving instructions for a specific application. In this notebook, that prior exposure may help the model recognize formal nineteenth-century wording, rhetorical repetition, parallel structure, metaphorical language, and relationships among ideas in Lincoln’s speech.

Large-scale pretraining can support fluent interpretation and summarization, but it does not guarantee that a response is historically accurate. A GPT-like model generates text by predicting plausible language from the supplied prompt and patterns learned during training. It does not independently verify each statement against archival evidence, and its output may combine direct information from the source text with associations acquired during pretraining. As a result, a response may be coherent and contextually appropriate while still omitting nuance, overstating an interpretation, or introducing historical details not contained in the speech.

This distinction is important because fluency can create an appearance of reliability. Lin et al. (2022) found that language models can produce confident and plausible responses that reflect common misconceptions or falsehoods. Their findings reinforce the need to evaluate generated language rather than assume that coherence is evidence of truthfulness. For this reason, the National Park Service version of the *Gettysburg Address* serves as the primary reference for assessing the model’s interpretations and summaries. Claims supported directly by the speech are distinguished from reasonable interpretation and from external historical information requiring additional verification.

The experiments in this notebook therefore evaluate more than whether the model can produce readable text. They examine whether the model can preserve the speech’s meaning, explain formal language accurately, identify central themes, follow summarization constraints, and limit unsupported content. Prompt comparisons and repeated trials are also used to examine whether the responses remain consistent when the wording of the instructions or the individual generation changes.

Transformer-based attention and generative pretraining provide a strong technical foundation for explanation, paraphrasing, theme identification, and summarization. However, the executed outputs must still be interpreted cautiously. Source grounding, structured evaluation, and human review remain necessary because convincing language does not independently demonstrate complete historical understanding, factual reliability, or consistent performance across other documents.

# 2. Python Environment, Secure API Configuration, and Reproducibility

This section prepares the computational environment used for the Gettysburg Address experiments. The notebook imports the libraries required for API communication, text processing, tabular analysis, timing, and output display. Establishing the environment before generating responses helps ensure that later results can be reproduced, reviewed, and interpreted consistently.

The selected model is defined once and used consistently throughout the experiments. This centralized configuration supports reproducibility and prevents differences among outputs from being unintentionally introduced by switching models.



In [ ]:
%pip -q install --upgrade openai

In [ ]:
import os
import re
import json
import time
from datetime import datetime, timezone
from difflib import SequenceMatcher
from getpass import getpass
from itertools import combinations

import pandas as pd
from IPython.display import display, Markdown
from openai import (
    OpenAI,
    OpenAIError,
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
)

pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


## Secure Credential Handling

The OpenAI API key is entered through a hidden prompt and stored only in the active Colab runtime as an environment variable. It is not written directly into a visible code cell, printed in the output, or saved as part of the notebook. Because the key exists only within the active runtime, it must be entered again if the Colab session is restarted or disconnected.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key (input is hidden): "
    ).strip()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("No API key was entered. Re-run this cell and provide a valid key.")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# GPT-like model selected for this focused text-analysis case study.
MODEL_NAME = "gpt-5-mini"

print("OpenAI client configured successfully.")
print(f"Selected model: {MODEL_NAME}")
print("The API key remains hidden and is not printed.")

OpenAI client configured successfully.
Selected model: gpt-5-mini
The API key remains hidden and is not printed.


# 3. Authoritative Source Text and Human Benchmark

The National Park Service version of Abraham Lincoln’s *Gettysburg Address* is used as the authoritative text for every source-grounded experiment in this notebook. The complete speech is stored in a Python variable so that the model receives the same document across the interpretation, theme-identification, formal-language, and summarization tasks.

Providing the full speech reduces dependence on the model’s pretrained memory and creates a consistent basis for evaluating whether its responses remain faithful to the source. It does not eliminate the possibility of unsupported generation, because the model may still combine the supplied text with associations learned during pretraining. Generated claims must therefore be compared directly with the speech before they are treated as supported.

The source metadata, paragraph count, word count, and character count are recorded to document the exact text used in the experiment. These descriptive values support reproducibility but do not measure the speech’s meaning or the model’s performance. Word counts may also vary slightly across tools because punctuation and hyphenated expressions can be tokenized differently.

In [ ]:
SOURCE_TITLE = "Gettysburg Address"
SOURCE_AUTHOR = "Abraham Lincoln"
SOURCE_DATE = "November 19, 1863"
SOURCE_URL = "https://www.nps.gov/liho/learn/historyculture/gettysburgaddress.htm"

gettysburg_address = """Four score and seven years ago our fathers brought forth upon this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal.

Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived and so dedicated, can long endure. We are met on a great battlefield of that war. We have come to dedicate a portion of that field, as a final resting place for those who here gave their lives that that nation might live. It is altogether fitting and proper that we should do this.

But, in a larger sense, we can not dedicate - we can not consecrate - we can not hallow - this ground. The brave men, living and dead, who struggled here have consecrated it, far above our poor power to add or detract. The world will little note, nor long remember what we say here, but it can never forget what they did here. It is for us the living, rather, to be dedicated here to the unfinished work which they who fought here have thus far so nobly advanced.

It is rather for us to be here dedicated to the great task remaining before us - that from these honored dead we take increased devotion to that cause for which they gave the last full measure of devotion - that we here highly resolve that these dead shall not have died in vain - that this nation, under God, shall have a new birth of freedom - and that government of the people, by the people, and for the people, shall not perish from the earth."""

paragraph_count = len([p for p in gettysburg_address.split("\n\n") if p.strip()])
word_count = len(re.findall(r"\b[\w’'-]+\b", gettysburg_address))
character_count = len(gettysburg_address)

print(f"Source: {SOURCE_AUTHOR}, {SOURCE_TITLE} ({SOURCE_DATE})")
print(f"National Park Service URL: {SOURCE_URL}")
print(f"Paragraph count: {paragraph_count}")
print(f"Word count: {word_count}")
print(f"Character count: {character_count}")

Source: Abraham Lincoln, Gettysburg Address (November 19, 1863)
National Park Service URL: https://www.nps.gov/liho/learn/historyculture/gettysburgaddress.htm
Paragraph count: 4
Word count: 272
Character count: 1473


In [ ]:
display(
    Markdown(
        f"### {SOURCE_TITLE}\n\n"
        f"*{SOURCE_AUTHOR}, {SOURCE_DATE}*\n\n"
        + gettysburg_address
    )
)

### Gettysburg Address

*Abraham Lincoln, November 19, 1863*

Four score and seven years ago our fathers brought forth upon this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal.

Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived and so dedicated, can long endure. We are met on a great battlefield of that war. We have come to dedicate a portion of that field, as a final resting place for those who here gave their lives that that nation might live. It is altogether fitting and proper that we should do this.

But, in a larger sense, we can not dedicate - we can not consecrate - we can not hallow - this ground. The brave men, living and dead, who struggled here have consecrated it, far above our poor power to add or detract. The world will little note, nor long remember what we say here, but it can never forget what they did here. It is for us the living, rather, to be dedicated here to the unfinished work which they who fought here have thus far so nobly advanced.

It is rather for us to be here dedicated to the great task remaining before us - that from these honored dead we take increased devotion to that cause for which they gave the last full measure of devotion - that we here highly resolve that these dead shall not have died in vain - that this nation, under God, shall have a new birth of freedom - and that government of the people, by the people, and for the people, shall not perish from the earth.

## Human Benchmark

Before reviewing the model-generated outputs, the speech was examined directly to establish a transparent human benchmark. Lincoln begins by connecting the nation’s founding with liberty and equality. He then presents the Civil War as a test of whether a nation established on those principles can endure.

The speech honors those who died at Gettysburg, but its purpose extends beyond remembrance. Lincoln argues that the soldiers’ actions have already consecrated the battlefield more meaningfully than ceremonial words could. Responsibility therefore shifts to the living, who must continue the “unfinished work,” ensure that the soldiers did not die in vain, and pursue a “new birth of freedom.”

The conclusion connects sacrifice and renewed national purpose with the survival of democratic government. Based on this reading, the central themes used as an interpretive benchmark are equality, sacrifice and remembrance, national endurance, freedom, civic responsibility, and government accountable to the people.

This benchmark is not presented as the only possible interpretation of the speech. It identifies the principal ideas that are directly visible in the text and provides a consistent basis for evaluating whether the model preserves those ideas, omits important meaning, or introduces unsupported content.

# 4. Reusable API Workflow and Evaluation Utilities

This section establishes the standardized workflow used to generate and document the model’s responses. A reusable request function applies the same model configuration and response-extraction process across the passage-interpretation, formal-language, theme-identification, summarization, and repeated-trial experiments. Centralizing this procedure reduces unnecessary variation in the implementation and supports consistent comparison among outputs.

For every successful request, the function records the experiment label, selected model, UTC timestamp, response identifier, request latency, prompt length, output word count, and generated text. This information creates a transparent experiment log and helps ensure that later interpretations remain aligned with the responses that were actually produced.

The function also reports authentication, quota, connection, and other API failures explicitly. Technical failures are not treated as evidence of weak language-model performance because they do not measure historical fidelity, contextual relevance, or summarization quality. Requests are not automatically retried, which limits unnecessary API usage and prevents failed attempts from being silently replaced by later responses.

Additional utility functions support word-count verification, response display, lexical comparison, and theme screening. These automated checks provide descriptive evidence, but they do not independently determine whether an interpretation is historically accurate or meaningful. Human review against the authoritative source remains necessary.

In [ ]:
api_run_log = []


def count_words(text):
    """Count word-like tokens for transparent length checks."""
    return len(re.findall(r"\b[\w’'-]+\b", str(text).strip()))


def normalize_words(text):
    """Return lowercase word tokens for lexical comparisons."""
    return re.findall(r"\b[\w’'-]+\b", str(text).lower())


def generate_gpt_response(
    prompt,
    experiment_label,
    max_output_tokens=700,
    instructions=None,
):
    """Send one prompt through the Responses API and log the result."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("The prompt must be a nonempty string.")
    if not isinstance(experiment_label, str) or not experiment_label.strip():
        raise ValueError("The experiment label must be a nonempty string.")

    request_arguments = {
        "model": MODEL_NAME,
        "input": prompt.strip(),
        "max_output_tokens": max_output_tokens,
        "reasoning": {"effort": "minimal"},
        "store": False,
    }
    if instructions:
        request_arguments["instructions"] = instructions.strip()

    started = time.perf_counter()

    try:
        response = client.responses.create(**request_arguments)
        elapsed_seconds = time.perf_counter() - started
        response_text = (response.output_text or "").strip()

        if not response_text:
            raise RuntimeError("The API returned no extractable text output.")

        run_record = {
            "experiment": experiment_label.strip(),
            "model": MODEL_NAME,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "response_id": getattr(response, "id", None),
            "latency_seconds": round(elapsed_seconds, 3),
            "prompt_characters": len(prompt.strip()),
            "output_words": count_words(response_text),
            "output_text": response_text,
        }
        api_run_log.append(run_record)
        return response_text

    except AuthenticationError as error:
        print("Authentication failed. Confirm that the API key is valid and active.")
        print(f"Technical detail: {error}")
    except RateLimitError as error:
        print("The request was rate-limited or the account has insufficient quota.")
        print(f"Technical detail: {error}")
    except APIConnectionError as error:
        print("The API connection failed. Check the internet connection and try again.")
        print(f"Technical detail: {error}")
    except OpenAIError as error:
        print("The API rejected the request.")
        print(f"Technical detail: {error}")
    except Exception as error:
        print(f"Unexpected error: {type(error).__name__}: {error}")

    return None


def show_response(title, response_text):
    """Display a generated response with a heading and word count."""
    if response_text is None:
        display(Markdown(f"### {title}\n\n*No successful response was returned.*"))
        return
    display(
        Markdown(
            f"### {title}\n\n{response_text}\n\n"
            f"**Word count:** {count_words(response_text)}"
        )
    )


def lexical_jaccard(text_a, text_b):
    """Compute transparent word-set overlap between two texts."""
    set_a = set(normalize_words(text_a))
    set_b = set(normalize_words(text_b))
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


print("Reusable generation and evaluation functions created successfully.")

Reusable generation and evaluation functions created successfully.


## Evaluation Boundaries

The utilities created in this section standardize data collection and provide transparent descriptive measurements. Word counts can verify length requirements, while lexical-overlap measures can show whether repeated responses use similar vocabulary. However, these measures cannot determine whether the model preserved historical meaning, interpreted rhetoric correctly, or avoided unsupported claims. Those questions require direct comparison with the *Gettysburg Address* and evidence-based human interpretation.

# 5. Source-Grounded Interpretation of the Assigned Passage

This experiment evaluates the model's explanation of the opening passage of the *Gettysburg Address*. The selected excerpt introduces the nation’s founding, liberty, and equality, while also establishing ideas that Lincoln develops throughout the remainder of the speech.

The complete speech is supplied with the prompt so that the model’s explanation can be evaluated against a consistent authoritative source. The prompt requires the model to translate the formal expression “four score and seven years ago,” explain the significance of liberty and equality, and connect the opening passage with the speech’s broader purpose.

The response is also instructed to distinguish claims supported directly by the speech from broader historical context. This distinction allows the analysis to evaluate not only fluency and clarity, but also historical fidelity, interpretive depth, instruction compliance, and control of unsupported information.

In [ ]:
assigned_excerpt = (
    "Four score and seven years ago our fathers brought forth upon this continent, "
    "a new nation, conceived in liberty, and dedicated to the proposition that all "
    "men are created equal."
)

assigned_prompt = f"""Use the supplied Gettysburg Address as the primary source for your response.

FULL SPEECH:
{gettysburg_address}

PASSAGE TO INTERPRET:
"{assigned_excerpt}"

Write a clear interpretation of approximately 250 to 350 words. Organize the response
into four numbered parts:

1. Explain what "four score and seven years ago" means in modern language.
2. Explain what Lincoln means by a nation "conceived in liberty."
3. Explain the significance of dedication to the proposition that all people are created equal.
4. Explain how the passage establishes the larger purpose of the Gettysburg Address.

Base the explanation primarily on the supplied speech. Clearly label any broader historical
context that is not stated directly in the text. Do not invent quotations, dates, people, or events.
Return a complete response without an additional introduction or conclusion."""

assigned_interpretation = generate_gpt_response(
    prompt=assigned_prompt,
    experiment_label="Opening-passage interpretation",
    max_output_tokens=1800,
)

show_response(
    "Generated Interpretation of the Assigned Passage",
    assigned_interpretation
)

### Generated Interpretation of the Assigned Passage

1. "Four score and seven years ago" in modern language means "eighty-seven years ago." Lincoln uses "score" as twenty years; four score (80) plus seven equals 87. He is pointing back to a specific founding moment 87 years prior, framing the origin of the United States as a distinct event in the past that set the stage for the present situation he addresses.

2. By a nation "conceived in liberty," Lincoln likens the nation's origin to the birth of a person conceived with liberty as its guiding principle. He emphasizes that the nation's founding idea was freedom: the political and moral intention that individual freedom and self-determination would be central. The phrase suggests that liberty was not an accidental byproduct but the deliberate, defining purpose at the nation's creation.

3. Dedication to the proposition that "all men are created equal" signals a moral and political commitment at the nation's founding. Lincoln treats this proposition as a foundational principle to which the nation was deliberately bound. Within the speech itself he elevates this proposition as the cause worth defending and as the standard by which current sacrifices (the battlefield dead) gain meaning. Labeling this as "dedicated" implies obligation: the living must honor and continue the work implied by that equality. (Broader historical context not stated directly in the text: Lincoln is referencing the Declaration of Independence’s language and using it as the touchstone for the nation’s purpose.)

4. The passage sets the larger purpose of the Gettysburg Address by establishing the stakes and moral frame for the Civil War. By recalling the nation's founding principles—liberty and equality—Lincoln connects the soldiers' sacrifice at Gettysburg to the preservation and fulfillment of those original commitments. He reframes the battlefield dedication into a renewed national dedication: to ensure that the nation, grounded in liberty and equality, endures and that government of, by, and for the people survives. The passage thus turns a local funeral into a call to national purpose and moral renewal.

**Word count:** 330

## Interpretation of the Assigned Passage

The generated interpretation successfully addressed all four parts of the prompt and remained within the requested 250–350-word range at 330 words. It correctly translated “four score and seven years ago” as 87 years and accurately explained that Lincoln uses the opening to connect the nation’s founding with liberty and equality. The response also recognized the birth metaphor in “conceived in liberty” and explained that liberty is presented as a defining principle rather than an incidental feature of the nation’s creation.

The response preserved the speech’s larger progression by connecting the opening principles with sacrifice, responsibility of the living, national endurance, and the survival of “government of, by, and for the people.” This interpretation is consistent with the supplied text, which moves from the nation’s founding ideals to the Civil War as a test of whether that nation can endure and concludes with renewed dedication to freedom and democratic government (National Park Service, 2015). The numbered structure was clear, coherent, and directly aligned with the requested task.

The response also followed the evidence-boundary instruction by explicitly labeling its reference to the Declaration of Independence as broader historical context not stated directly in the speech. However, some additional wording extended beyond the source without the same qualification. Descriptions such as “individual freedom and self-determination” represent reasonable interpretation rather than language stated directly in the address. The final description of the occasion as a “local funeral” is also imprecise because the supplied speech refers to dedicating part of the battlefield as a final resting place rather than describing the event as a funeral.

Overall, the response demonstrated strong source-grounded interpretation, organization, contextual relevance, and instruction compliance. Its principal limitation was not a misunderstanding of the passage, but incomplete separation of direct textual evidence from interpretive expansion. Because this assessment concerns one response to one historical passage, it does not establish that the model would interpret other historical texts with the same accuracy, nuance, or evidence control.

# 6. Historical and Formal-Language Analysis

This section evaluates how the GPT-like model handles selected expressions from the *Gettysburg Address* that may be less familiar to modern readers. The four passages were selected because they represent different challenges in historical-text interpretation: formal vocabulary, rhetorical repetition, metaphorical language, implicit meaning, and parallel structure.

For each passage, the model is asked to provide a modern-language paraphrase, explain the expression within the context of the complete speech, and identify its rhetorical function. The prompt also requires the model to distinguish direct textual meaning from broader historical interpretation. This design allows the responses to be evaluated for more than readability; it examines whether the model preserves Lincoln’s meaning, recognizes how language contributes to the speech’s purpose, and avoids presenting unsupported context as direct evidence.

The same response structure and approximate length requirement are applied across all four prompts. This consistency supports comparison among the outputs, although the passages differ in complexity and should not be expected to produce identical levels of detail.

In [ ]:
formal_language_tasks = [
    {
        "label": "Consecrate and hallow",
        "passage": (
            "We can not dedicate - we can not consecrate - "
            "we can not hallow - this ground."
        ),
        "focus": (
            "Explain the distinctions among dedicate, consecrate, and hallow, "
            "and analyze how repetition strengthens Lincoln's argument."
        ),
    },
    {
        "label": "Unfinished work",
        "passage": (
            "It is for us the living, rather, to be dedicated here to the "
            "unfinished work which they who fought here have thus far so "
            "nobly advanced."
        ),
        "focus": (
            "Explain what unfinished work means within the speech and how "
            "Lincoln shifts responsibility from the fallen soldiers to the living."
        ),
    },
    {
        "label": "New birth of freedom",
        "passage": (
            "This nation, under God, shall have a new birth of freedom."
        ),
        "focus": (
            "Explain the metaphor of a new birth and how it connects freedom "
            "with renewed national purpose."
        ),
    },
    {
        "label": "Government of the people",
        "passage": (
            "Government of the people, by the people, and for the people."
        ),
        "focus": (
            "Explain the three-part parallel structure and how it expresses "
            "Lincoln's view of democratic government."
        ),
    },
]

formal_language_results = []

for item in formal_language_tasks:
    prompt = f"""Use the supplied Gettysburg Address as the primary textual evidence.

FULL SPEECH:
{gettysburg_address}

PASSAGE TO ANALYZE:
"{item['passage']}"

ANALYTICAL FOCUS:
{item['focus']}

Write a response of approximately 130 to 180 words using these three labeled parts:

1. Modern Paraphrase
2. Textual and Rhetorical Analysis
3. Evidence Boundary

Preserve the passage's meaning and explain its relationship to the complete speech.
In the Evidence Boundary section, identify whether the explanation is directly supported
by the text, represents a reasonable interpretation, or introduces broader historical
context requiring external verification. Do not invent quotations, dates, people, or events."""

    response_text = generate_gpt_response(
        prompt=prompt,
        experiment_label=f"Formal language: {item['label']}",
        max_output_tokens=1000,
    )

    response_word_count = (
        count_words(response_text) if response_text else None
    )

    formal_language_results.append(
        {
            "label": item["label"],
            "passage": item["passage"],
            "response": response_text,
            "word_count": response_word_count,
            "within_requested_range": (
                130 <= response_word_count <= 180
                if response_word_count is not None
                else False
            ),
        }
    )

for result in formal_language_results:
    show_response(result["label"], result["response"])

### Consecrate and hallow

1. Modern Paraphrase
Lincoln says that we cannot make this ground special by ceremony or words alone; the soldiers’ actions have already made it sacred beyond our ability to add to or take away from that sanctity.

2. Textual and Rhetorical Analysis
The three verbs move from a common civic act (dedicate) to more serious religious terms (consecrate, hallow), sharpening meaning with each step. "Dedicate" suggests a public commitment or setting aside; "consecrate" implies formal religious blessing; "hallow" evokes ultimate sacredness. The triple negative ("we can not") plus the parallel structure creates a rising emotional and ethical force that emphasizes human insufficiency. Repetition slows the cadence, invites reflection, and contrasts human words with soldierly sacrifice. In the context of the whole speech, this claim redirects honor from oratory and ceremony to the dead’s deeds, supporting Lincoln’s call that the living take up the unfinished work.

3. Evidence Boundary
Most claims (word meanings, repetition’s rhetorical effect, contrast with soldiers’ actions, relation to call for dedication) are directly supported by the text and reasonable rhetorical interpretation. Broader linguistic nuances of each verb and psychological effects of repetition are interpretive but consistent with the speech; they do not require external historical facts.

**Word count:** 199

### Unfinished work

1. Modern Paraphrase
Lincoln is saying that the living—not the dead—must continue the unfinished work begun by those who fought and died. The soldiers have already advanced the cause; now the survivors must commit themselves to completing that mission.

2. Textual and Rhetorical Analysis
"Unfinished work" refers back to the speech’s themes: preserving a nation "conceived in liberty" and realizing equality. Grammatically and rhetorically, Lincoln shifts agency with contrast: he contrasts what the dead have done (“they who fought here have thus far so nobly advanced”) with what remains for "us the living" to do. The anaphora of dedication earlier and the solemn tone convert mourning into a civic duty. By framing the task as continued devotion, Lincoln moves responsibility from passive reverence for sacrifice to active political and moral effort—completing the promise of the Republic so the dead’s sacrifice has meaning.

3. Evidence Boundary
Directly supported by the text: the shift of responsibility from the fallen to "us the living" and the call to be "dedicated...to the unfinished work." Reasonable interpretation: that the "unfinished work" means securing the nation’s principles of liberty and equality. Broader historical context (specific policies, events, or outcomes tied to that work) would require external verification.

**Word count:** 205

### New birth of freedom

1. Modern Paraphrase
"This country, with God’s guidance, will experience a renewed beginning of true freedom," meaning the nation will be reborn with a deeper commitment to liberty and equality.

2. Textual and Rhetorical Analysis
The phrase "new birth" is a compact metaphor likening the nation to a living organism that can be born again. Birth implies renewal, a formative fresh start rather than mere repair. Tying "new birth" to "freedom" compresses moral and political renewal into a single image: emancipation and recommitment to the Founding principle that "all men are created equal." Placed after assertions that the dead gave the "last full measure of devotion" and that the living must carry forward "unfinished work," the line rhetorically converts sacrifice into promise. The religious phrase "under God" sanctifies the renewal, lending moral authority and linking civic duty to higher purpose. The metaphor thus connects personal sacrifice, national healing, and a renewed public commitment to liberty.

3. Evidence Boundary
This explanation is grounded in the speech’s language and structure (directly supported) and includes reasonable rhetorical interpretation of metaphor and placement within the address. No external historical claims are introduced.

**Word count:** 188

### Government of the people

1. Modern Paraphrase
"Government of the people, by the people, and for the people" means the state belongs to ordinary citizens, is run through their participation or representatives, and exists to serve their interests.

2. Textual and Rhetorical Analysis
Lincoln uses a three-part parallel (anaphora and tricolon) that tightens rhythm and emphasizes completeness. Each phrase shifts focus—origination ("of the people"), agency ("by the people"), and purpose ("for the people")—creating a full picture of democratic legitimacy. Placed at the speech’s close, this cadence transforms the funeral context into a universal political claim: the sacrifice on the battlefield affirms a republic rooted in popular sovereignty and devoted to human equality. The balanced repetition makes the idea memorable and morally weighty, linking the Civil War’s "unfinished work" to preserving a government grounded in the people rather than elites.

3. Evidence Boundary
Directly supported by the text: the three phrases and their placement as the speech’s concluding idea. Reasonable interpretation: the functions (origin, agency, purpose) and rhetorical effect. Broader context requiring external verification: linking this line to later democratic theory or Lincoln’s broader political practice beyond the speech.

**Word count:** 186

In [ ]:
formal_language_df = pd.DataFrame(formal_language_results)

display(
    formal_language_df[
        [
            "label",
            "word_count",
            "within_requested_range",
        ]
    ]
)

,label,word_count,within_requested_range
0,Consecrate and hallow,199,False
1,Unfinished work,205,False
2,New birth of freedom,188,False
3,Government of the people,186,False


## Interpretation of the Historical and Formal-Language Responses

All four responses were complete, clearly organized, and directly relevant to their opening passages. Each used the requested three-part structure of modern paraphrase, textual and rhetorical analysis, and evidence classification. However, none followed the requested 130–180-word range. The responses contained 199, 205, 188, and 186 words, respectively. Therefore, 0 of 4 responses met the length requirement. This is a consistent instruction-compliance weakness even though the additional length generally supported detailed analysis.

The response to “dedicate,” “consecrate,” and “hallow” accurately explained how the sequence moves from setting the ground apart toward increasingly sacred language. It also recognized that the repeated phrase “we can not” shifts attention away from ceremonial words and toward the soldiers’ actions. Its connection between this contrast and the living’s responsibility to continue the unfinished work was consistent with the complete speech. The response appropriately classified the finer distinctions among the three verbs and the emotional effects of repetition as interpretation rather than external historical fact.

The “unfinished work” response clearly identified Lincoln’s shift from honoring the fallen to assigning responsibility to “us the living.” It accurately connected that responsibility with national endurance, liberty, equality, and continued dedication. Its distinction between direct textual support and reasonable interpretation was also effective. However, the statement that the work must be continued from those who “fought and died” slightly narrows the passage’s wording, which directly refers to those who “fought here” while the surrounding speech separately honors those who gave their lives. This did not materially change the central meaning, but it illustrates how paraphrasing can blend nearby ideas.

The “new birth of freedom” response correctly recognized birth as a metaphor for national renewal and connected the phrase with sacrifice, unfinished work, liberty, and equality. Its evidence classification was less reliable. The response introduced “emancipation,” described “under God” as sanctifying the renewal, and referred to “God’s guidance,” although these explanations extend beyond what the speech states directly. It then concluded that no external historical claims had been introduced. In addition, the modern paraphrase was placed in quotation marks even though it was not Lincoln’s exact wording. These issues demonstrate incomplete separation of source evidence, interpretation, and broader historical context.

The “government of the people” response accurately explained the three-part parallel structure as expressing governmental origin, agency, and purpose. It also connected the conclusion with democratic government and the preservation of a nation grounded in equality. However, calling the construction “anaphora” was imprecise because the repeated wording appears at the ends of the parallel phrases rather than at their beginnings. The references to representatives, popular sovereignty, elites, and a “funeral context” also extend beyond the wording of the supplied speech and should be identified more clearly as interpretation or broader context.

Overall, the model handled formal vocabulary, metaphor, repetition, shifts in responsibility, and parallel structure with substantial interpretive strength. The principal weaknesses were consistent failure to meet the requested length, occasional imprecision in rhetorical terminology, quotation drift, and incomplete evidence-boundary labeling. These findings support the use of GPT-like models for preliminary historical-language explanation, but they also demonstrate the need for direct source comparison and human review before accepting the analysis as historically or rhetorically precise.

# 7. Source-Grounded Identification of Key Themes

This section evaluates whether the GPT-like model can synthesize the central themes of the *Gettysburg Address* while remaining grounded in the supplied text. Theme identification requires more than locating repeated words because several of Lincoln’s main ideas are expressed through metaphor, rhetorical structure, and relationships developed across the speech.

The model is asked to identify six central themes, explain each theme concisely, and support each interpretation with a brief phrase from the address. Requiring textual evidence creates a transparent basis for evaluating whether the themes are supported directly by the speech or depend on broader interpretation.

The generated analysis will be compared with the initial human benchmark established in Section 3. The comparison considers theme coverage, fidelity to the source, quality of textual support, interpretive depth, and control of unsupported historical content. Differences from the human benchmark are not automatically treated as errors because themes may be combined, divided, or described using different terminology.

In [ ]:
theme_prompt = f"""Use the supplied Gettysburg Address as the primary source.

FULL SPEECH:
{gettysburg_address}

Identify six central themes in the speech.

For each theme, use the following structure:

1. Theme:
2. Explanation:
3. Textual Evidence:
4. Evidence Classification:

Requirements:
- Give each theme a concise and descriptive label.
- Explain each theme in approximately 45 to 70 words.
- Include one brief phrase quoted exactly from the supplied speech.
- Classify the explanation as either directly supported by the text or a reasonable interpretation.
- Do not introduce historical facts, dates, events, or people that are not stated in the speech.
- Do not repeat the same theme using different wording.

Conclude with a short synthesis of approximately 50 to 75 words explaining how
the six themes work together to communicate the speech's overall purpose."""

theme_response = generate_gpt_response(
    prompt=theme_prompt,
    experiment_label="Source-grounded key-theme identification",
    max_output_tokens=1800,
)

show_response("Generated Theme Analysis", theme_response)

### Generated Theme Analysis

1. Theme: Equality as Founding Principle
2. Explanation: The speech presents equality as the fundamental idea that gave rise to the nation and grounds its moral purpose. This principle is framed as the reason the nation was "conceived" and as the standard by which the nation’s endurance and sacrifices should be judged and renewed.
3. Textual Evidence: "conceived in liberty, and dedicated to the proposition that all men are created equal."
4. Evidence Classification: Directly supported by the text

1. Theme: National Endurance Tested by War
2. Explanation: The address frames the civil conflict as a severe test of whether a nation built on its founding principles can survive. The war’s purpose is portrayed as demonstrating whether such a nation “can long endure,” emphasizing trial and existential evaluation.
3. Textual Evidence: "testing whether that nation, or any nation so conceived and so dedicated, can long endure."
4. Evidence Classification: Directly supported by the text

1. Theme: Commemoration of Sacrifice
2. Explanation: The speech centers on honoring those who died on the battlefield, portraying the dedication of the ground as a means to remember and respect their ultimate sacrifice. The dead are depicted as having given their lives so the nation might live, warranting the ceremony.
3. Textual Evidence: "a final resting place for those who here gave their lives that that nation might live."
4. Evidence Classification: Directly supported by the text

1. Theme: Limits of Ceremonial Acts
2. Explanation: The speaker argues that formal acts of dedication cannot truly consecrate the ground because the soldiers’ actions have already sanctified it. This theme emphasizes the insufficiency of words and ceremonies compared with the moral power of deeds.
3. Textual Evidence: "we can not dedicate - we can not consecrate - we can not hallow - this ground."
4. Evidence Classification: Directly supported by the text

1. Theme: Duty of the Living to Continue the Work
2. Explanation: The address shifts responsibility to the living, urging them to remain committed to the unfinished tasks of those who fought. The living must be “dedicated” to advancing the cause and ensuring the purpose of the sacrifices is carried forward.
3. Textual Evidence: "It is for us the living, rather, to be dedicated here to the unfinished work which they who fought here have thus far so nobly advanced."
4. Evidence Classification: Directly supported by the text

1. Theme: Resolve for National Renewal and Democratic Survival
2. Explanation: The speech culminates in a call for renewed devotion that aims to achieve a "new birth of freedom" and secure the survival of a government rooted in popular sovereignty. The living’s resolve is tied to preventing the nation’s democratic principles from perishing.
3. Textual Evidence: "that this nation, under God, shall have a new birth of freedom - and that government of the people, by the people, and for the people, shall not perish from the earth."
4. Evidence Classification: Directly supported by the text

Synthesis:
Together these themes move from origin (equality) through trial (war) and memorial (commemoration, limits of ceremony) to responsibility (duty of the living) and resolve (renewal and democratic survival). They form a coherent appeal: honor the dead by committing the living to sustain the nation’s founding principles so that the sacrifices achieve enduring national purpose.

**Word count:** 540

In [ ]:
theme_patterns = {
    "Equality and founding principles": [
        r"\ball (?:people|men) are created equal\b",
        r"\bcreated equal\b",
        r"\bequality\b",
        r"\bproposition\b",
    ],
    "Sacrifice and remembrance": [
        r"\bgave their lives\b",
        r"\bhonored dead\b",
        r"\blast full measure of devotion\b",
        r"\bsacrifice\b",
    ],
    "National endurance and preservation": [
        r"\bcan long endure\b",
        r"\bnation might live\b",
        r"\bshall not perish\b",
        r"\bnational endurance\b",
    ],
    "Freedom and liberty": [
        r"\bnew birth of freedom\b",
        r"\bfreedom\b",
        r"\bliberty\b",
    ],
    "Responsibility of the living": [
        r"\bunfinished work\b",
        r"\bgreat task remaining\b",
        r"\bfor us the living\b",
        r"\bcivic responsibility\b",
    ],
    "Democratic government": [
        r"\bgovernment of the people\b",
        r"\bby the people\b",
        r"\bfor the people\b",
        r"\bdemocratic government\b",
        r"\bdemocracy\b",
    ],
}


def theme_coverage(text):
    """Perform a transparent lexical screen for benchmark themes."""
    text_lower = str(text).lower()
    rows = []

    for theme, patterns in theme_patterns.items():
        matched_patterns = [
            pattern
            for pattern in patterns
            if re.search(pattern, text_lower)
        ]

        rows.append(
            {
                "benchmark_theme": theme,
                "lexical_evidence_detected": bool(matched_patterns),
                "matched_patterns": (
                    ", ".join(matched_patterns)
                    if matched_patterns
                    else "None"
                ),
            }
        )

    return pd.DataFrame(rows)


theme_coverage_df = theme_coverage(theme_response or "")
display(theme_coverage_df)

themes_detected = int(
    theme_coverage_df["lexical_evidence_detected"].sum()
)

print(
    f"Benchmark themes detected lexically: "
    f"{themes_detected} of {len(theme_patterns)}"
)

,benchmark_theme,lexical_evidence_detected,matched_patterns
0,Equality and founding principles,True,"\ball (?:people|men) are created equal\b, \bcreated equal\b, \bequality\b, \bproposition\b"
1,Sacrifice and remembrance,True,"\bgave their lives\b, \bsacrifice\b"
2,National endurance and preservation,True,"\bcan long endure\b, \bnation might live\b, \bshall not perish\b, \bnational endurance\b"
3,Freedom and liberty,True,"\bnew birth of freedom\b, \bfreedom\b, \bliberty\b"
4,Responsibility of the living,True,"\bunfinished work\b, \bfor us the living\b"
5,Democratic government,True,"\bgovernment of the people\b, \bby the people\b, \bfor the people\b"


Benchmark themes detected lexically: 6 of 6


## Interpretive Limits of the Automated Theme Check

The automated table is a lexical screening tool rather than a semantic-accuracy measure. It identifies whether selected words or phrases associated with the human benchmark appear in the generated analysis. A theme may be explained accurately using different terminology and therefore remain undetected, while the presence of a matching phrase does not prove that the model interpreted it correctly.

The automated results must therefore be evaluated together with the complete response. Human review is required to determine whether each theme is distinct, supported by an appropriate quotation, explained accurately, and connected meaningfully with the speech’s overall purpose.

## Interpretation of the Key-Theme Analysis

The model generated six distinct themes and concluded with a coherent synthesis. The automated lexical screen detected all 6 benchmark themes established in Section 3: equality and founding principles, sacrifice and remembrance, national endurance, freedom and liberty, responsibility of the living, and democratic government. This result indicates broad thematic coverage, although lexical detection alone does not establish that every theme was interpreted accurately.

The generated themes did not correspond one-to-one with the human benchmark. Freedom and democratic government were combined into “Resolve for National Renewal and Democratic Survival,” while “Limits of Ceremonial Acts” was presented as a separate theme. This was a reasonable organizational choice because the speech directly contrasts ceremonial language with the soldiers’ actions. The synthesis then connected the speech’s progression from founding principles, through war and commemoration, to the responsibilities of the living and the survival of democratic government.

The response used source-based textual evidence for every theme. The quotation-fidelity screen examined eight quoted phrases in the theme analysis, and all eight matched wording in the supplied speech after normalization. The source-fidelity screen also detected no selected external historical context. These findings support strong quotation accuracy and source grounding. However, several evidence excerpts contained 26 or 31 words, making them longer than the prompt’s request for a brief phrase.

Instruction compliance was mixed. The six explanations contained 46, 40, 44, 37, 39, and 42 words. Only the first explanation met the requested 45–70-word range, so 1 of 6 theme explanations satisfied the specified length. The 54-word synthesis did meet its requested 50–75-word range. Despite the shorter explanations, the response remained understandable and covered the principal meaning of each theme.

The evidence classifications were the main interpretive weakness. All six explanations were labeled “directly supported by the text,” even though several included reasonable interpretation. Descriptions such as equality grounding the nation’s “moral purpose,” ceremonial language being insufficient compared with the “moral power of deeds,” and government being rooted in “popular sovereignty” are defensible interpretations, but those exact conclusions are not stated directly in the speech. These themes should have been classified more carefully as combinations of textual support and reasonable inference.

Overall, the model demonstrated strong thematic synthesis, accurate quotation use, broad benchmark coverage, and effective organization. Its principal weaknesses were inconsistent compliance with the requested explanation lengths and an overly broad use of the “directly supported” classification. The results show that automated theme detection and exact quotation matching can support evaluation, but human review remains necessary to distinguish direct evidence from interpretation.

# 8. Fifty-Word Summarization and Prompt-Design Comparison

This section evaluates whether the GPT-like model can compress the *Gettysburg Address* into exactly 50 words while preserving its central meaning. Summarization is challenging because the model must balance brevity with coverage of equality, sacrifice, national purpose, freedom, responsibility of the living, and democratic government.

Three prompting conditions are evaluated:

1. **Minimal basic prompt:** Reproduces the brief summarization instruction without supplying the speech.
2. **Source-provided prompt:** Supplies the complete speech but gives only a basic length instruction.
3. **Source-grounded constrained prompt:** Supplies the speech and explicitly identifies the themes, evidence boundary, output format, and exact word-count requirement.

This progression allows the outputs to be compared for word-count compliance, thematic coverage, source fidelity, clarity, and unsupported content. However, the experiment is descriptive rather than causal. Because the conditions differ in more than one way, any improvement cannot be attributed solely to one prompt feature. Repeated trials and additional controlled comparisons would be needed for stronger conclusions.

In [ ]:
# Condition 1: Minimal prompt without the source text.
basic_summary_prompt = "Summarize the Gettysburg Address in 50 words."

basic_summary = generate_gpt_response(
    prompt=basic_summary_prompt,
    experiment_label="Minimal basic 50-word summary",
    max_output_tokens=500,
)


# Condition 2: Basic instruction with the authoritative source supplied.
source_provided_summary_prompt = f"""Summarize the following Gettysburg Address
in exactly 50 words.

FULL SPEECH:
{gettysburg_address}

Return only the summary without a title, label, explanation, or word-count statement."""

source_provided_summary = generate_gpt_response(
    prompt=source_provided_summary_prompt,
    experiment_label="Source-provided 50-word summary",
    max_output_tokens=500,
)


# Condition 3: Source-grounded prompt with explicit thematic constraints.
grounded_summary_prompt = f"""Using only the Gettysburg Address supplied below,
write an exactly 50-word summary.

FULL SPEECH:
{gettysburg_address}

The summary should preserve the speech's central ideas of:

- equality and the nation's founding principles,
- sacrifice and remembrance,
- national endurance,
- freedom,
- responsibility of the living, and
- democratic government.

Do not introduce facts, dates, people, or events that are not stated in the speech.
Return only the summary without a title, label, explanation, bullet points, or
word-count statement."""

grounded_summary = generate_gpt_response(
    prompt=grounded_summary_prompt,
    experiment_label="Source-grounded constrained 50-word summary",
    max_output_tokens=500,
)


show_response("Minimal Basic Summary", basic_summary)
show_response("Source-Provided Summary", source_provided_summary)
show_response("Source-Grounded Constrained Summary", grounded_summary)

### Course-Style Basic Summary

Four score and seven years after America's founding, Lincoln affirmed the Union's purpose: preserve a nation dedicated to equality and liberty. He honored fallen soldiers at Gettysburg, urged living citizens to continue their sacrifice by completing the work of freedom, and pledged that government of, by, and for the people shall endure.

**Word count:** 52

### Source-Provided Summary

Lincoln honors soldiers’ sacrifice at Gettysburg, recalling the nation’s founding ideals of liberty and equality. He says the battlefield is already consecrated by their deeds and urges the living to continue their unfinished work: to rededicate themselves to freedom so the nation endures and government of, by, and for the people survives.

**Word count:** 52

### Source-Grounded Constrained Summary

A nation conceived in liberty and dedicated to equality faces a great civil war testing its endurance; we honor those who sacrificed their lives, acknowledging they consecrated this ground, and we, the living, must commit to unfinished work, renewed devotion to freedom, ensuring government of, by, and for the people endures.

**Word count:** 51

In [ ]:
summary_comparison = pd.DataFrame(
    [
        {
            "prompt_condition": "Minimal basic",
            "source_text_supplied": False,
            "explicit_theme_guidance": False,
            "word_count": (
                count_words(basic_summary)
                if basic_summary
                else None
            ),
            "exactly_50_words": (
                count_words(basic_summary) == 50
                if basic_summary
                else False
            ),
            "summary": basic_summary,
        },
        {
            "prompt_condition": "Source-provided",
            "source_text_supplied": True,
            "explicit_theme_guidance": False,
            "word_count": (
                count_words(source_provided_summary)
                if source_provided_summary
                else None
            ),
            "exactly_50_words": (
                count_words(source_provided_summary) == 50
                if source_provided_summary
                else False
            ),
            "summary": source_provided_summary,
        },
        {
            "prompt_condition": "Source-grounded constrained",
            "source_text_supplied": True,
            "explicit_theme_guidance": True,
            "word_count": (
                count_words(grounded_summary)
                if grounded_summary
                else None
            ),
            "exactly_50_words": (
                count_words(grounded_summary) == 50
                if grounded_summary
                else False
            ),
            "summary": grounded_summary,
        },
    ]
)

display(
    summary_comparison[
        [
            "prompt_condition",
            "source_text_supplied",
            "explicit_theme_guidance",
            "word_count",
            "exactly_50_words",
        ]
    ]
)

,prompt_condition,source_text_supplied,explicit_theme_guidance,word_count,exactly_50_words
0,Minimal basic,False,False,52,False
1,Source-provided,True,False,52,False
2,Source-grounded constrained,True,True,51,False


## Interpretive Limits of the Summary Checks

Exact word count is an objective measure of instruction compliance, but it does not establish that a summary is accurate, complete, or historically faithful. Similarly, the lexical theme screen identifies selected words and phrases associated with the human benchmark, but it cannot determine whether those ideas were interpreted meaningfully.

A 50-word limit also creates an unavoidable compression tradeoff. A response may accurately emphasize several themes while omitting others because the requested length does not allow every idea to receive equal attention. The final evaluation must therefore consider which themes were prioritized, whether essential meaning was preserved, and whether any unsupported information was introduced.

In [ ]:
summary_outputs = {
    "Minimal basic": basic_summary,
    "Source-provided": source_provided_summary,
    "Source-grounded constrained": grounded_summary,
}

summary_theme_rows = []

for condition, summary_text in summary_outputs.items():
    coverage = theme_coverage(summary_text or "")

    detected_themes = coverage.loc[
        coverage["lexical_evidence_detected"],
        "benchmark_theme",
    ].tolist()

    summary_theme_rows.append(
        {
            "prompt_condition": condition,
            "benchmark_themes_detected": len(detected_themes),
            "benchmark_themes_checked": len(coverage),
            "detected_themes": (
                ", ".join(detected_themes)
                if detected_themes
                else "None detected"
            ),
        }
    )

summary_theme_comparison = pd.DataFrame(summary_theme_rows)
display(summary_theme_comparison)

,prompt_condition,benchmark_themes_detected,benchmark_themes_checked,detected_themes
0,Minimal basic,4,6,"Equality and founding principles, Sacrifice and remembrance, Freedom and liberty, Democratic government"
1,Source-provided,5,6,"Equality and founding principles, Sacrifice and remembrance, Freedom and liberty, Responsibility of the living, Demo..."
2,Source-grounded constrained,4,6,"Equality and founding principles, Freedom and liberty, Responsibility of the living, Democratic government"


## Interpretation of the Summarization and Prompt-Design Results

All three prompting conditions produced complete, coherent summaries that preserved the speech’s main purpose. However, none followed the exact 50-word requirement. The minimal basic and source-provided summaries each contained 52 words, while the source-grounded constrained summary contained 51 words. Therefore, 0 of 3 outputs achieved exact length compliance. The small overages did not substantially reduce readability, but they demonstrate that an explicit numerical instruction did not guarantee exact compliance.

The minimal basic summary accurately addressed the nation’s founding ideals, the soldiers’ sacrifice, the responsibility of the living, freedom, national preservation, and democratic government. Because the prompt did not supply the speech, the response relied more heavily on pretrained knowledge. It introduced “the Union,” Lincoln’s name, and Gettysburg as contextual details. The source-fidelity screen specifically flagged “the Union” because that term does not appear in the supplied speech. These additions are historically plausible, but they extend beyond the document itself and demonstrate why a response generated from model memory requires separate verification.

The source-provided summary remained closely aligned with the speech. It connected liberty and equality with sacrifice, consecration of the battlefield, unfinished work, freedom, national endurance, and government “of, by, and for the people.” No selected external-context terms were detected. The response was readable and source-faithful, although its 52-word length exceeded the requested limit by two words.

The source-grounded constrained summary also preserved the major progression of the address: a nation founded on liberty and equality is tested by civil war, the fallen are honored, the living inherit responsibility for unfinished work, freedom requires renewed dedication, and democratic government must endure. It contained no selected external-context flags and exceeded the length requirement by only one word. Its compressed sentence structure was somewhat denser than the other summaries, but the central meaning remained understandable and consistent with the authoritative text (National Park Service, 2015).

The automated lexical screen detected 4 of 6 benchmark themes in the minimal basic summary, 5 of 6 in the source-provided summary, and 4 of 6 in the constrained summary. These totals understated the actual thematic content. For example, the constrained summary referred to people who “sacrificed their lives” and a war “testing [the nation’s] endurance,” but those phrases did not match the exact lexical patterns used for sacrifice and national endurance. Direct human review showed that all three summaries represented the six benchmark areas at least briefly, although they used different wording and emphasis.

The source-provided condition received the highest automated theme count, while the more detailed constrained prompt did not produce either exact 50-word compliance or a higher lexical score. This does not establish that thematic guidance was ineffective. The lexical tool was sensitive to wording, and each condition produced only one response. Because the prompts also differed in source availability, thematic instructions, and formatting constraints, the observed differences cannot be attributed causally to any single prompt feature.

Overall, supplying the authoritative speech reduced reliance on external historical context and created a clearer basis for source comparison. Explicit thematic guidance supported broad conceptual coverage, but it did not guarantee exact length compliance or a higher automated theme score. The experiment demonstrates that prompt design can influence the form and grounding of a summary, while human review remains necessary to evaluate meaning, source fidelity, and compliance accurately.

# 9. Repeated-Trial Consistency and Output Variability

This section evaluates whether the source-grounded constrained summary remains consistent when the same prompt is submitted to the same model multiple times. The original source-grounded summary from Section 8 is compared with three additional responses generated under the same notebook configuration.

The analysis examines three forms of consistency:

1. **Instruction compliance:** Whether each output contains exactly 50 words.
2. **Thematic stability:** Whether the benchmark themes identified in Section 3 remain represented across trials.
3. **Language variability:** Whether the summaries use identical wording or express similar ideas through different phrasing.

Repeated outputs do not need to be identical to be accurate. Different wording may represent acceptable paraphrasing, while similar wording does not independently demonstrate historical fidelity. The generated summaries must therefore be evaluated through both descriptive measurements and direct comparison with the authoritative source text.

This is a limited consistency test rather than a general reliability study. Four total outputs from one prompt, one speech, and one model cannot estimate the full range of possible responses. Model updates, API behavior, and additional trials could produce different results.

In [ ]:
if "grounded_summary" not in globals() or not grounded_summary:
    raise RuntimeError(
        "Run the source-grounded constrained summary in Section 8 before Section 9."
    )

repeat_summaries = []

for trial_number in range(1, 4):
    response_text = generate_gpt_response(
        prompt=grounded_summary_prompt,
        experiment_label=(
            f"Repeated source-grounded constrained summary trial {trial_number}"
        ),
        max_output_tokens=500,
    )

    repeat_summaries.append(
        {
            "trial_number": trial_number,
            "trial_label": f"Repeated trial {trial_number}",
            "summary": response_text,
        }
    )

# Retain the three additional trials for later notebook summaries.
repeat_summaries_df = pd.DataFrame(repeat_summaries)

# Combine the original Section 8 output with the three repeated calls.
all_grounded_trials = [
    {
        "trial_number": 0,
        "trial_label": "Initial Section 8 output",
        "summary": grounded_summary,
    }
] + repeat_summaries

for record in all_grounded_trials:
    summary_text = record["summary"]

    if summary_text:
        coverage = theme_coverage(summary_text)

        detected_themes = coverage.loc[
            coverage["lexical_evidence_detected"],
            "benchmark_theme",
        ].tolist()

        record["word_count"] = count_words(summary_text)
        record["exactly_50_words"] = count_words(summary_text) == 50
        record["benchmark_themes_detected"] = len(detected_themes)
        record["detected_themes"] = (
            ", ".join(detected_themes)
            if detected_themes
            else "None detected"
        )
    else:
        record["word_count"] = None
        record["exactly_50_words"] = False
        record["benchmark_themes_detected"] = 0
        record["detected_themes"] = "No response returned"

all_grounded_trials_df = pd.DataFrame(all_grounded_trials)

display(
    all_grounded_trials_df[
        [
            "trial_label",
            "word_count",
            "exactly_50_words",
            "benchmark_themes_detected",
            "detected_themes",
        ]
    ]
)

# Display only the three new outputs because the initial summary
# was already displayed in Section 8.
for record in repeat_summaries:
    show_response(record["trial_label"], record["summary"])

,trial_label,word_count,exactly_50_words,benchmark_themes_detected,detected_themes
0,Initial Section 8 output,51,False,4,"Equality and founding principles, Freedom and liberty, Responsibility of the living, Democratic government"
1,Repeated trial 1,54,False,5,"Equality and founding principles, Sacrifice and remembrance, Freedom and liberty, Responsibility of the living, Demo..."
2,Repeated trial 2,51,False,5,"Equality and founding principles, Sacrifice and remembrance, Freedom and liberty, Responsibility of the living, Demo..."
3,Repeated trial 3,54,False,5,"Equality and founding principles, Sacrifice and remembrance, Freedom and liberty, Responsibility of the living, Demo..."


### Repeated trial 1

A nation founded on liberty and equality faces civil war testing its endurance. Honoring those who died consecrates the field; their sacrifice demands remembrance and renewed devotion. The living must continue their unfinished work so the dead did not perish in vain, ensuring a reborn freedom and government of, by, and for the people.

**Word count:** 54

### Repeated trial 2

A nation founded on equality faces a civil trial; brave dead consecrate the battlefield through sacrifice. We honor and remember them, committing the living to continue their unfinished work, ensuring their deaths were not in vain, securing a renewed freedom and the endurance of government of, by, and for the people.

**Word count:** 51

### Repeated trial 3

A nation founded on liberty and equality faces a grave civil war testing its endurance. Honoring those who consecrated the battlefield by sacrifice, we must dedicate ourselves to unfinished work: increase devotion to their cause, ensure their deaths are not in vain, birth renewed freedom, and preserve government of, by, and for the people.

**Word count:** 54

In [ ]:
valid_trials = [
    (record["trial_label"], record["summary"])
    for record in all_grounded_trials
    if record["summary"]
]

similarity_rows = []

for (label_a, text_a), (label_b, text_b) in combinations(
    valid_trials,
    2,
):
    similarity_rows.append(
        {
            "comparison": f"{label_a} vs. {label_b}",
            "exact_text_match": text_a.strip() == text_b.strip(),
            "sequence_similarity": round(
                SequenceMatcher(
                    None,
                    text_a.lower(),
                    text_b.lower(),
                ).ratio(),
                3,
            ),
            "lexical_jaccard": round(
                lexical_jaccard(text_a, text_b),
                3,
            ),
            "word_count_difference": abs(
                count_words(text_a) - count_words(text_b)
            ),
        }
    )

pairwise_similarity_df = pd.DataFrame(similarity_rows)

if pairwise_similarity_df.empty:
    print(
        "Not enough successful trials were available "
        "for pairwise comparison."
    )
else:
    display(pairwise_similarity_df)

,comparison,exact_text_match,sequence_similarity,lexical_jaccard,word_count_difference
0,Initial Section 8 output vs. Repeated trial 1,False,0.350,0.475,3
1,Initial Section 8 output vs. Repeated trial 2,False,0.174,0.387,0
2,Initial Section 8 output vs. Repeated trial 3,False,0.294,0.492,3
3,Repeated trial 1 vs. Repeated trial 2,False,0.205,0.467,3
4,Repeated trial 1 vs. Repeated trial 3,False,0.257,0.550,0
5,Repeated trial 2 vs. Repeated trial 3,False,0.231,0.459,3


In [ ]:
theme_presence_rows = []

for record in all_grounded_trials:
    if not record["summary"]:
        continue

    coverage = theme_coverage(record["summary"])

    for _, theme_row in coverage.iterrows():
        theme_presence_rows.append(
            {
                "trial_label": record["trial_label"],
                "benchmark_theme": theme_row["benchmark_theme"],
                "detected": bool(
                    theme_row["lexical_evidence_detected"]
                ),
            }
        )

theme_presence_df = pd.DataFrame(theme_presence_rows)

if theme_presence_df.empty:
    print(
        "No successful summaries were available "
        "for theme-stability analysis."
    )
else:
    theme_stability_matrix = theme_presence_df.pivot(
        index="benchmark_theme",
        columns="trial_label",
        values="detected",
    )

    display(theme_stability_matrix)

    theme_stability_summary = (
        theme_presence_df.groupby(
            "benchmark_theme",
            as_index=False,
        )
        .agg(
            trials_detected=("detected", "sum"),
            successful_trials=("detected", "count"),
        )
    )

    theme_stability_summary["detection_rate"] = (
        theme_stability_summary["trials_detected"]
        / theme_stability_summary["successful_trials"]
    ).round(3)

    display(theme_stability_summary)

trial_label,Initial Section 8 output,Repeated trial 1,Repeated trial 2,Repeated trial 3
benchmark_theme,,,,
Democratic government,True,True,True,True
Equality and founding principles,True,True,True,True
Freedom and liberty,True,True,True,True
National endurance and preservation,False,False,False,False
Responsibility of the living,True,True,True,True
Sacrifice and remembrance,False,True,True,True


,benchmark_theme,trials_detected,successful_trials,detection_rate
0,Democratic government,4,4,1.00
1,Equality and founding principles,4,4,1.00
2,Freedom and liberty,4,4,1.00
3,National endurance and preservation,0,4,0.00
4,Responsibility of the living,4,4,1.00
5,Sacrifice and remembrance,3,4,0.75


In [ ]:
successful_trials_df = all_grounded_trials_df.dropna(
    subset=["summary"]
).copy()

successful_trial_count = len(successful_trials_df)

if successful_trial_count:
    exact_50_count = int(
        successful_trials_df["exactly_50_words"].sum()
    )

    exact_50_rate = round(
        exact_50_count / successful_trial_count,
        3,
    )

    mean_word_count = round(
        successful_trials_df["word_count"].mean(),
        2,
    )

    minimum_word_count = int(
        successful_trials_df["word_count"].min()
    )

    maximum_word_count = int(
        successful_trials_df["word_count"].max()
    )

    unique_output_count = int(
        successful_trials_df["summary"].nunique()
    )
else:
    exact_50_count = 0
    exact_50_rate = None
    mean_word_count = None
    minimum_word_count = None
    maximum_word_count = None
    unique_output_count = 0

mean_sequence_similarity = (
    round(
        pairwise_similarity_df[
            "sequence_similarity"
        ].mean(),
        3,
    )
    if not pairwise_similarity_df.empty
    else None
)

mean_lexical_jaccard = (
    round(
        pairwise_similarity_df[
            "lexical_jaccard"
        ].mean(),
        3,
    )
    if not pairwise_similarity_df.empty
    else None
)

variability_summary_df = pd.DataFrame(
    [
        {
            "measure": "Successful grounded-summary outputs",
            "result": successful_trial_count,
        },
        {
            "measure": "Outputs meeting exact 50-word requirement",
            "result": exact_50_count,
        },
        {
            "measure": "Exact 50-word compliance rate",
            "result": exact_50_rate,
        },
        {
            "measure": "Mean word count",
            "result": mean_word_count,
        },
        {
            "measure": "Observed word-count range",
            "result": (
                f"{minimum_word_count} to {maximum_word_count}"
                if successful_trial_count
                else None
            ),
        },
        {
            "measure": "Unique output texts",
            "result": unique_output_count,
        },
        {
            "measure": "Mean pairwise sequence similarity",
            "result": mean_sequence_similarity,
        },
        {
            "measure": "Mean pairwise lexical Jaccard similarity",
            "result": mean_lexical_jaccard,
        },
    ]
)

display(variability_summary_df)

,measure,result
0,Successful grounded-summary outputs,4
1,Outputs meeting exact 50-word requirement,0
2,Exact 50-word compliance rate,0.0
3,Mean word count,52.5
4,Observed word-count range,51 to 54
5,Unique output texts,4
6,Mean pairwise sequence similarity,0.252
7,Mean pairwise lexical Jaccard similarity,0.472


## Interpretive Limits of the Variability Measures

Sequence similarity measures character-level resemblance, while lexical Jaccard similarity measures overlap among the unique words used in two summaries. These measures describe wording consistency but do not determine historical accuracy, thematic completeness, or interpretive quality.

An exact match may indicate stable generation, but repeated wording does not prove that the content is correct. Conversely, lower similarity may reflect acceptable paraphrasing rather than a meaningful inconsistency. The thematic-stability analysis is also limited because it relies on selected lexical patterns. A theme may be expressed accurately using wording that the automated screen does not recognize.

The results must therefore be interpreted through direct review of the complete summaries, their relationship to the authoritative source, their exact word counts, and the themes emphasized or omitted. Because the analysis includes only four outputs, all conclusions remain descriptive and limited to this experiment.

## Interpretation of Repeated-Trial Consistency

All four source-grounded summary requests returned complete responses, but none followed the exact 50-word requirement. The initial Section 8 output contained 51 words, repeated trial 1 contained 54 words, repeated trial 2 contained 51 words, and repeated trial 3 contained 54 words. The mean length was 52.5 words, with an observed range of 51–54 words. Therefore, the exact 50-word compliance rate was 0%. This repeated pattern shows that the model consistently produced concise responses while remaining slightly above the required limit.

The four summaries preserved the speech’s central progression despite using different wording. Each described a nation founded on liberty or equality, a civil war testing its endurance, the sacrifice of those who fought or died, the responsibility of the living to continue unfinished work, renewed freedom, and the survival of government “of, by, and for the people.” Repeated trial 2 used the phrase “civil trial,” which was less precise than the speech’s wording “great civil war,” but the remainder of the sentence preserved the intended idea that the nation was being tested.

The automated lexical screen detected four benchmark themes in the initial output and five in each repeated trial. Equality and founding principles, freedom and liberty, responsibility of the living, and democratic government were detected in all four outputs. Sacrifice and remembrance was detected in three of four trials, while national endurance and preservation was not detected in any trial.

These automated results understated the actual thematic stability. The initial summary referred to people who “sacrificed their lives,” and all four summaries described the nation, war, or government as enduring or being tested for endurance. Those ideas were not always recognized because the generated wording did not match the limited lexical patterns used by the screening function. Direct review therefore indicated that all six benchmark areas were represented across the four summaries, even though the automated table did not record them consistently.

All four outputs were unique, and none was an exact text match. Pairwise sequence-similarity values ranged from 0.174 to 0.350, with a mean of 0.252. Lexical Jaccard similarity ranged from 0.387 to 0.550, with a mean of 0.472. These results indicate substantial variation in sentence construction and moderate overlap in vocabulary. The model did not reproduce a fixed response, but the central meaning remained comparatively stable across the trials.

The repeated outputs also varied in emphasis. Trial 1 stressed remembrance and renewed devotion, trial 2 emphasized commitment by the living, and trial 3 used more compressed language about sacrifice, unfinished work, and renewed freedom. These differences represent acceptable paraphrasing rather than contradictory interpretations. However, the repeated failure to satisfy the exact word-count requirement demonstrates that thematic stability did not guarantee complete instruction compliance.

Overall, the experiment showed stronger consistency in meaning than in wording or exact length. The model repeatedly preserved the address’s main themes while changing phrasing and emphasis, but it failed the precise 50-word constraint in every trial. Because the analysis includes only four outputs from one prompt and one model, these findings describe the executed experiment and should not be generalized as a broader estimate of model reliability.

# 10. Source-Fidelity and External-Verification Screening

This section examines whether the generated outputs remain grounded in the supplied *Gettysburg Address* or introduce information that extends beyond the speech. The purpose is not to classify every external statement as inaccurate. A historically correct statement may still require verification when it is not supported directly by the document being analyzed.

Two complementary screening procedures are used:

1. **External-context screening:** Identifies selected references to historical documents, slavery, emancipation, military organizations, battle details, cemetery-dedication details, and constitutional amendments that are not stated directly in the speech.
2. **Quotation-fidelity screening:** Extracts phrases presented in quotation marks and checks whether they match wording in the authoritative source text after basic normalization.

These automated procedures support transparency, but they cannot independently determine whether a broader historical claim is true, whether a paraphrase is reasonable, or whether an interpretation is meaningful. Flagged excerpts require human review and, when appropriate, verification through an authoritative historical source. The absence of a flag should not be interpreted as proof that an entire response is accurate.

In [ ]:
# Assemble the current successful outputs without requiring
# every optional section to exist.
named_outputs = {}

candidate_outputs = {
    "Opening-passage interpretation": globals().get(
        "assigned_interpretation"
    ),
    "Theme analysis": globals().get("theme_response"),
    "Minimal basic summary": globals().get(
        "basic_summary"
    ),
    "Source-provided summary": globals().get(
        "source_provided_summary"
    ),
    "Source-grounded constrained summary": globals().get(
        "grounded_summary"
    ),
}

for output_name, output_text in candidate_outputs.items():
    if isinstance(output_text, str) and output_text.strip():
        named_outputs[output_name] = output_text.strip()


# Add the four formal-language responses.
if "formal_language_results" in globals():
    for item in formal_language_results:
        output_text = item.get("response")

        if isinstance(output_text, str) and output_text.strip():
            label = item.get("label", "Unnamed passage")

            named_outputs[
                f"Formal-language analysis: {label}"
            ] = output_text.strip()


# Add the three additional repeated-summary trials.
if "repeat_summaries" in globals():
    for item in repeat_summaries:
        output_text = item.get("summary")

        if isinstance(output_text, str) and output_text.strip():
            trial_label = item.get(
                "trial_label",
                f"Repeated trial "
                f"{item.get('trial_number', '')}".strip(),
            )

            named_outputs[trial_label] = output_text.strip()


# Remove exact duplicate outputs while preserving
# the first descriptive label.
unique_named_outputs = {}
seen_texts = set()

for output_name, output_text in named_outputs.items():
    normalized_text = re.sub(
        r"\s+",
        " ",
        output_text,
    ).strip().lower()

    if normalized_text not in seen_texts:
        unique_named_outputs[output_name] = output_text
        seen_texts.add(normalized_text)

named_outputs = unique_named_outputs

print(
    "Successful unique outputs included in Section 10: "
    f"{len(named_outputs)}"
)

In [ ]:
external_context_patterns = {
    "Declaration of Independence or 1776": [
        r"\b1776\b",
        r"\bdeclaration of independence\b",
    ],
    "Slavery, abolition, or emancipation": [
        r"\bslaver(?:y|ies)?\b",
        r"\babolition\w*\b",
        r"\bemancipat\w*\b",
    ],
    "Union or Confederacy": [
        r"\bunion\b",
        r"\bconfedera\w*\b",
    ],
    "Battle-specific details": [
        r"\bbattle of gettysburg\b",
        r"\bcasualt\w*\b",
        r"\bjuly\s+[1-3](?:st|nd|rd)?\b",
    ],
    "Cemetery-dedication details": [
        r"\bsoldiers'? national cemetery\b",
        r"\bcemetery\b",
    ],
    "Constitutional amendments": [
        r"\b13th amendment\b",
        r"\bthirteenth amendment\b",
        r"\b14th amendment\b",
        r"\bfourteenth amendment\b",
        r"\b15th amendment\b",
        r"\bfifteenth amendment\b",
    ],
}


def split_review_units(text):
    """Split generated text into readable units for review."""
    units = re.split(
        r"(?<=[.!?])\s+|\n+",
        str(text).strip(),
    )

    return [
        unit.strip()
        for unit in units
        if unit.strip()
    ]


external_claim_rows = []

for output_name, output_text in named_outputs.items():
    for review_unit in split_review_units(output_text):
        review_unit_lower = review_unit.lower()

        for category, patterns in (
            external_context_patterns.items()
        ):
            matched_patterns = [
                pattern
                for pattern in patterns
                if re.search(
                    pattern,
                    review_unit_lower,
                )
            ]

            if matched_patterns:
                external_claim_rows.append(
                    {
                        "output": output_name,
                        "verification_category": category,
                        "flagged_excerpt": review_unit,
                        "review_status": (
                            "Requires external verification; "
                            "not automatically classified "
                            "as false"
                        ),
                    }
                )


external_claims_df = pd.DataFrame(
    external_claim_rows
)

screening_summary_rows = []

for output_name in named_outputs:
    flag_count = (
        int(
            (
                external_claims_df["output"]
                == output_name
            ).sum()
        )
        if not external_claims_df.empty
        else 0
    )

    screening_summary_rows.append(
        {
            "output": output_name,
            "external_context_flags": flag_count,
            "screening_result": (
                "Review flagged excerpts"
                if flag_count
                else (
                    "No selected external-context "
                    "terms detected"
                )
            ),
        }
    )


external_screening_summary_df = pd.DataFrame(
    screening_summary_rows
)

display(external_screening_summary_df)

if external_claims_df.empty:
    print(
        "No selected external-context terms were "
        "detected. Human review is still required."
    )
else:
    display(external_claims_df)

,output,external_context_flags,screening_result
0,Opening-passage interpretation,1,Review flagged excerpts
1,Theme analysis,0,No selected external-context terms detected
2,Minimal basic summary,1,Review flagged excerpts
3,Source-provided summary,0,No selected external-context terms detected
4,Source-grounded constrained summary,0,No selected external-context terms detected
5,Formal-language analysis: Consecrate and hallow,0,No selected external-context terms detected
6,Formal-language analysis: Unfinished work,0,No selected external-context terms detected
7,Formal-language analysis: New birth of freedom,1,Review flagged excerpts
8,Formal-language analysis: Government of the people,0,No selected external-context terms detected
9,Repeated trial 1,0,No selected external-context terms detected


,output,verification_category,flagged_excerpt,review_status
0,Opening-passage interpretation,Declaration of Independence or 1776,(Broader historical context not stated directly in the text: Lincoln is referencing the Declaration of Independence’...,Requires external verification; not automatically classified as false
1,Minimal basic summary,Union or Confederacy,"Four score and seven years after America's founding, Lincoln affirmed the Union's purpose: preserve a nation dedicat...",Requires external verification; not automatically classified as false
2,Formal-language analysis: New birth of freedom,"Slavery, abolition, or emancipation","Tying ""new birth"" to ""freedom"" compresses moral and political renewal into a single image: emancipation and recommit...",Requires external verification; not automatically classified as false


In [ ]:
def normalize_for_source_comparison(text):
    """Normalize text for transparent source matching."""
    normalized = str(text).lower()

    normalized = (
        normalized
        .replace("’", "'")
        .replace("‘", "'")
        .replace("—", " ")
        .replace("–", " ")
        .replace("-", " ")
    )

    normalized = re.sub(
        r"[^a-z0-9'\s]",
        " ",
        normalized,
    )

    return re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()


def extract_double_quoted_phrases(text):
    """Extract straight or curly double-quoted phrases."""
    matches = re.findall(
        r'[“"]([^"”]{3,300})[”"]',
        str(text),
    )

    return [
        match.strip()
        for match in matches
        if match.strip()
    ]


normalized_source = normalize_for_source_comparison(
    gettysburg_address
)

quotation_rows = []

for output_name, output_text in named_outputs.items():
    quoted_phrases = extract_double_quoted_phrases(
        output_text
    )

    for quoted_phrase in quoted_phrases:
        quoted_word_count = count_words(
            quoted_phrase
        )

        # Very short quoted labels are not treated
        # as documentary quotations.
        if quoted_word_count < 3:
            continue

        normalized_quote = (
            normalize_for_source_comparison(
                quoted_phrase
            )
        )

        exact_source_match = (
            normalized_quote in normalized_source
        )

        quotation_rows.append(
            {
                "output": output_name,
                "quoted_phrase": quoted_phrase,
                "quoted_words": quoted_word_count,
                "exact_source_match_after_normalization": (
                    exact_source_match
                ),
                "review_status": (
                    "Exact source phrase detected"
                    if exact_source_match
                    else (
                        "Review as paraphrase, altered "
                        "quotation, or external wording"
                    )
                ),
            }
        )


quotation_fidelity_df = pd.DataFrame(
    quotation_rows
)

if quotation_fidelity_df.empty:
    print(
        "No quoted phrases of three or more words "
        "were detected for automated source matching."
    )
else:
    display(quotation_fidelity_df)

,output,quoted_phrase,quoted_words,exact_source_match_after_normalization,review_status
0,Opening-passage interpretation,Four score and seven years ago,6,True,Exact source phrase detected
1,Opening-passage interpretation,eighty-seven years ago.,3,False,"Review as paraphrase, altered quotation, or external wording"
2,Opening-passage interpretation,"conceived in liberty,",3,True,Exact source phrase detected
3,Opening-passage interpretation,all men are created equal,5,True,Exact source phrase detected
4,Theme analysis,"conceived in liberty, and dedicated to the proposition that all men are created equal.",14,True,Exact source phrase detected
5,Theme analysis,"can long endure,",3,True,Exact source phrase detected
6,Theme analysis,"testing whether that nation, or any nation so conceived and so dedicated, can long endure.",15,True,Exact source phrase detected
7,Theme analysis,a final resting place for those who here gave their lives that that nation might live.,16,True,Exact source phrase detected
8,Theme analysis,we can not dedicate - we can not consecrate - we can not hallow - this ground.,14,True,Exact source phrase detected
9,Theme analysis,"It is for us the living, rather, to be dedicated here to the unfinished work which they who fought here have thus fa...",26,True,Exact source phrase detected


In [ ]:
source_fidelity_rows = []

for output_name in named_outputs:
    external_flag_count = (
        int(
            (
                external_claims_df["output"]
                == output_name
            ).sum()
        )
        if not external_claims_df.empty
        else 0
    )

    if quotation_fidelity_df.empty:
        quotation_count = 0
        unmatched_quotation_count = 0
    else:
        output_quotes = quotation_fidelity_df[
            quotation_fidelity_df["output"]
            == output_name
        ]

        quotation_count = len(output_quotes)

        unmatched_quotation_count = int(
            (
                ~output_quotes[
                    "exact_source_match_after_normalization"
                ]
            ).sum()
        )

    source_fidelity_rows.append(
        {
            "output": output_name,
            "external_context_flags": (
                external_flag_count
            ),
            "quoted_phrases_checked": (
                quotation_count
            ),
            "quoted_phrases_not_matched_exactly": (
                unmatched_quotation_count
            ),
            "human_review_required": "Yes",
        }
    )


source_fidelity_summary_df = pd.DataFrame(
    source_fidelity_rows
)

display(source_fidelity_summary_df)

,output,external_context_flags,quoted_phrases_checked,quoted_phrases_not_matched_exactly,human_review_required
0,Opening-passage interpretation,1,4,1,Yes
1,Theme analysis,0,8,0,Yes
2,Minimal basic summary,1,0,0,Yes
3,Source-provided summary,0,0,0,Yes
4,Source-grounded constrained summary,0,0,0,Yes
5,Formal-language analysis: Consecrate and hallow,0,1,0,Yes
6,Formal-language analysis: Unfinished work,0,5,1,Yes
7,Formal-language analysis: New birth of freedom,1,3,1,Yes
8,Formal-language analysis: Government of the people,0,4,0,Yes
9,Repeated trial 1,0,0,0,Yes


## Interpretive Limits of the Source-Fidelity Screening

The external-context scan is intentionally selective. It can identify only the terms and patterns included in the screening dictionary, so an output may contain a different unsupported claim without being flagged. A detected term also does not demonstrate that the associated statement is inaccurate. It shows only that the statement extends beyond the supplied speech and should be verified separately.

The quotation check tests whether normalized quoted wording appears in the authoritative source. A phrase that does not match exactly may be an acceptable paraphrase, a shortened quotation, a punctuation variation not handled by the normalization procedure, or an altered quotation requiring correction. Human review is necessary to distinguish among these possibilities.

These procedures therefore function as screening tools rather than automated historical fact-checkers. Final judgments about source fidelity must be based on direct comparison with the complete speech and, when broader context is introduced, consultation of an appropriate authoritative historical source.

## Interpretation of Source Fidelity and External Verification

The screening procedures evaluated 12 unique generated outputs from the interpretation, theme, formal-language, summarization, and repeated-trial experiments. Three external-context excerpts were flagged across three outputs. The remaining nine outputs contained no terms matched by the selected screening patterns, although the absence of a flag does not prove that an output contains no unsupported or interpretive expansion.

The opening-passage interpretation was flagged for its reference to the Declaration of Independence. This broader context was appropriately identified within the response as information not stated directly in the supplied speech. The statement therefore extended beyond the primary source but did not misleadingly present the external information as direct textual evidence. It would still require an authoritative secondary source if retained in a formal historical analysis.

The minimal basic summary was flagged for the term “Union.” That concept does not appear in the supplied version of the *Gettysburg Address*, and the summary did not distinguish it as external context. The addition was historically plausible, but it demonstrates the risk of relying on pretrained model knowledge when the source document is not included in the prompt. By comparison, the source-provided and source-grounded summaries produced no selected external-context flags.

The response analyzing “new birth of freedom” was flagged for introducing “emancipation.” This was the most important evidence-boundary problem identified by the screening. The response connected the metaphor with emancipation and later stated that it had introduced no external historical claims. Because emancipation is not stated directly in the supplied speech, the response should have classified that connection as broader historical context requiring verification rather than as purely source-grounded analysis.

The quotation-fidelity procedure examined 25 quoted phrases. Twenty-two matched wording in the authoritative source after normalization, while three did not match exactly. The opening-passage interpretation’s phrase “eighty-seven years ago” was a clear modern-language translation of “four score and seven years ago,” so the mismatch did not represent a factual error. However, quotation marks could still make a paraphrase appear to be Lincoln’s exact wording unless it is labeled clearly.

The “unfinished work” response used the shortened phrase “dedicated...to the unfinished work.” Although the wording was derived from the speech, it did not appear as one exact continuous phrase in the source. The ellipsis signaled omission, but a more precise quotation would preserve the original wording or make the omitted material clearer.

The “new birth of freedom” response placed the sentence “This country, with God’s guidance, will experience a renewed beginning of true freedom” in quotation marks. That sentence was a modern paraphrase rather than an exact quotation from the speech. It also added “God’s guidance,” which interprets the phrase “under God” more strongly than the source itself. This output therefore combined quotation drift with interpretive expansion and should not be presented as direct textual evidence.

The theme analysis demonstrated the strongest quotation fidelity. All eight quoted phrases checked in that response matched the speech after normalization. The analyses of “consecrate and hallow” and “government of the people” also contained no unmatched quotations. These findings support generally strong source use while showing that quotation reliability varied by task.

The automated scan did not identify every statement requiring interpretation or verification. Terms such as “popular sovereignty,” references to representatives or elites, and descriptions of the event as a funeral were not included in the external-context dictionary. Human review was therefore still necessary to identify these unflagged expansions.

Overall, most generated content remained closely connected to the supplied speech, and 22 of 25 checked quotations matched the source exactly after normalization. The main weaknesses were the unqualified use of “Union,” the introduction of emancipation while claiming no external context had been added, and two paraphrases presented in quotation marks as though they might be exact source language. The results demonstrate that source-fidelity screening can identify important review targets, but final historical and quotation judgments still require direct comparison with the authoritative text.

# 11. Structured Human Evaluation of Generated Outputs

Automated word counts, lexical screens, similarity measures, and source-fidelity flags provide useful descriptive evidence, but they cannot determine whether a historical interpretation is accurate, nuanced, or appropriately qualified. This section therefore applies a structured human evaluation to the major generated outputs.

Each response is evaluated using six criteria:

| Evaluation criterion | Human-review focus |
|---|---|
| **Historical fidelity** | Whether the response preserves the meaning and ideas expressed in the supplied speech. |
| **Contextual relevance** | Whether the response directly and fully addresses its prompt. |
| **Task-specific quality** | Whether the response performs the particular task well, such as interpreting formal language, identifying themes, or preserving meaning during summarization. |
| **Clarity and coherence** | Whether the response is understandable, logically organized, and appropriately written. |
| **Unsupported-content control** | Whether claims extending beyond the speech are avoided, qualified, or identified as requiring external verification. |
| **Instruction compliance** | Whether the response follows the requested structure, scope, source restrictions, and word-count requirements. |

### Rating Scale

| Rating | Interpretation |
|---|---|
| **5 — Strong** | Accurate, complete, clearly supported, and appropriately qualified, with no material correction needed. |
| **4 — Good** | Strong overall, with a minor omission, wording issue, or need for limited qualification. |
| **3 — Adequate** | Generally relevant and reasonable, but incomplete, uneven, vague, or only partly supported. |
| **2 — Weak** | Contains a meaningful omission, misunderstanding, unsupported expansion, or instruction failure. |
| **1 — Poor** | Substantially inaccurate, off-task, unsupported, unclear, or noncompliant. |

Ratings must be supported by specific evidence from the executed output. Fluency alone is not sufficient for a high rating. Each evidence note should identify an exact strength, omission, unsupported statement, quotation issue, or instruction-compliance result.

In [15]:
import pandas as pd
from IPython.display import display


evaluation_targets = [
    {
        "output": "Opening-passage interpretation",
        "task_type": "Opening-passage interpretation",
        "task_specific_focus": (
            "Accuracy and depth of the explanation of founding, "
            "liberty, equality, and the speech's larger purpose"
        ),
        "historical_fidelity": 4,
        "contextual_relevance": 5,
        "task_specific_quality": 4,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 4,
        "instruction_compliance": 5,
        "evidence_note": (
            "The 330-word response addressed all four requested parts, "
            "correctly translated four score and seven years as 87 years, "
            "and connected liberty and equality with sacrifice, national "
            "endurance, and democratic government. It also identified the "
            "Declaration of Independence as broader historical context."
        ),
        "limitation_or_correction": (
            "The phrase local funeral was imprecise, and descriptions such "
            "as individual freedom and self-determination represented "
            "interpretive expansion that was not clearly labeled."
        ),
    },
    {
        "output": "Theme analysis",
        "task_type": "Key-theme identification",
        "task_specific_focus": (
            "Distinct theme coverage, accurate textual support, "
            "and meaningful synthesis"
        ),
        "historical_fidelity": 4,
        "contextual_relevance": 5,
        "task_specific_quality": 4,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 4,
        "instruction_compliance": 3,
        "evidence_note": (
            "The response produced six distinct themes, covered all 6 "
            "benchmark themes lexically, used eight quotations that matched "
            "the speech after normalization, and completed a 54-word synthesis."
        ),
        "limitation_or_correction": (
            "Only 1 of 6 explanations met the requested 45–70-word range. "
            "All themes were classified as directly supported even though "
            "several explanations included reasonable interpretation, such "
            "as moral purpose, moral power of deeds, and popular sovereignty."
        ),
    },
    {
        "output": "Formal-language analysis: Consecrate and hallow",
        "task_type": "Formal-language interpretation",
        "task_specific_focus": (
            "Distinctions among dedicate, consecrate, and hallow, "
            "including the function of repetition"
        ),
        "historical_fidelity": 5,
        "contextual_relevance": 5,
        "task_specific_quality": 5,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 5,
        "instruction_compliance": 3,
        "evidence_note": (
            "The response accurately distinguished dedicate, consecrate, "
            "and hallow, explained the repeated phrase we can not, and "
            "connected the rhetorical contrast between words and soldiers' "
            "actions with the unfinished work assigned to the living."
        ),
        "limitation_or_correction": (
            "The response contained 199 words and therefore exceeded the "
            "requested 130–180-word range. Some descriptions of emotional "
            "and psychological effects were reasonable rhetorical "
            "interpretations rather than direct textual statements."
        ),
    },
    {
        "output": "Formal-language analysis: Unfinished work",
        "task_type": "Formal-language interpretation",
        "task_specific_focus": (
            "Meaning of unfinished work and the shift of "
            "responsibility to the living"
        ),
        "historical_fidelity": 4,
        "contextual_relevance": 5,
        "task_specific_quality": 4,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 4,
        "instruction_compliance": 3,
        "evidence_note": (
            "The response correctly explained the shift of responsibility "
            "from those who fought to us the living and connected unfinished "
            "work with continued dedication to liberty, equality, and "
            "national preservation."
        ),
        "limitation_or_correction": (
            "The response contained 205 words and exceeded the requested "
            "range. The paraphrase stating that those who fought and died "
            "began the work blended two ideas from nearby portions of the "
            "speech. The shortened quotation dedicated...to the unfinished "
            "work did not match one continuous source phrase."
        ),
    },
    {
        "output": "Formal-language analysis: New birth of freedom",
        "task_type": "Metaphorical interpretation",
        "task_specific_focus": (
            "Interpretation of the birth metaphor and its connection "
            "with renewed national purpose"
        ),
        "historical_fidelity": 3,
        "contextual_relevance": 5,
        "task_specific_quality": 4,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 2,
        "instruction_compliance": 3,
        "evidence_note": (
            "The response correctly interpreted new birth as a metaphor "
            "for national renewal and connected it with sacrifice, unfinished "
            "work, liberty, equality, and renewed public commitment."
        ),
        "limitation_or_correction": (
            "The response contained 188 words and exceeded the requested "
            "range. It introduced emancipation and God's guidance, presented "
            "a modern paraphrase in quotation marks, and then incorrectly "
            "claimed that no external historical information had been added."
        ),
    },
    {
        "output": "Formal-language analysis: Government of the people",
        "task_type": "Rhetorical interpretation",
        "task_specific_focus": (
            "Explanation of parallel structure and democratic government"
        ),
        "historical_fidelity": 4,
        "contextual_relevance": 5,
        "task_specific_quality": 4,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 3,
        "instruction_compliance": 3,
        "evidence_note": (
            "The response accurately interpreted the three phrases as "
            "governmental origin, agency, and purpose and explained how "
            "parallel structure and cumulative repetition strengthen the "
            "speech's concluding statement about democratic government."
        ),
        "limitation_or_correction": (
            "The response contained 186 words and exceeded the requested "
            "range. Calling the construction anaphora was imprecise, and "
            "references to representatives, elites, popular sovereignty, "
            "and a funeral context extended beyond the supplied wording."
        ),
    },
    {
        "output": "Minimal basic summary",
        "task_type": "Fifty-word summarization",
        "task_specific_focus": (
            "Preservation of essential meaning under a strict "
            "length constraint without supplied source text"
        ),
        "historical_fidelity": 4,
        "contextual_relevance": 5,
        "task_specific_quality": 4,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 3,
        "instruction_compliance": 4,
        "evidence_note": (
            "The 52-word summary preserved the speech's central ideas of "
            "equality, liberty, sacrifice, responsibility of the living, "
            "freedom, national preservation, and democratic government."
        ),
        "limitation_or_correction": (
            "The response exceeded the exact 50-word requirement by two "
            "words. It also introduced Lincoln, Gettysburg, and the Union "
            "without identifying them as context beyond the wording of the "
            "speech supplied elsewhere in the notebook."
        ),
    },
    {
        "output": "Source-provided summary",
        "task_type": "Fifty-word summarization",
        "task_specific_focus": (
            "Preservation of essential meaning when the complete "
            "speech is supplied"
        ),
        "historical_fidelity": 5,
        "contextual_relevance": 5,
        "task_specific_quality": 5,
        "clarity_and_coherence": 5,
        "unsupported_content_control": 5,
        "instruction_compliance": 4,
        "evidence_note": (
            "The summary remained closely grounded in the supplied speech "
            "and preserved liberty, equality, sacrifice, consecration, "
            "unfinished work, freedom, national endurance, and government "
            "of, by, and for the people. No selected external-context terms "
            "were detected."
        ),
        "limitation_or_correction": (
            "The response contained 52 words and exceeded the exact "
            "50-word requirement by two words. No material correction to "
            "its historical content was otherwise required."
        ),
    },
    {
        "output": "Source-grounded constrained summary",
        "task_type": "Fifty-word summarization",
        "task_specific_focus": (
            "Theme preservation, source grounding, and exact "
            "word-count compliance under explicit constraints"
        ),
        "historical_fidelity": 5,
        "contextual_relevance": 5,
        "task_specific_quality": 5,
        "clarity_and_coherence": 4,
        "unsupported_content_control": 5,
        "instruction_compliance": 4,
        "evidence_note": (
            "The summary preserved the progression from liberty and equality "
            "through civil war, sacrifice, unfinished work, renewed freedom, "
            "and democratic government. It produced no selected "
            "external-context flags and remained faithful to the speech."
        ),
        "limitation_or_correction": (
            "The response contained 51 words and exceeded the exact "
            "50-word requirement by one word. Its single compressed sentence "
            "was somewhat dense, although the central meaning remained clear."
        ),
    },
]


# All nine outputs were successfully generated and reviewed
# in the executed notebook.
available_evaluation_targets = evaluation_targets.copy()
missing_evaluation_targets = []

human_evaluation = pd.DataFrame(
    available_evaluation_targets
)


rating_columns = [
    "historical_fidelity",
    "contextual_relevance",
    "task_specific_quality",
    "clarity_and_coherence",
    "unsupported_content_control",
    "instruction_compliance",
]


# Store ratings in a nullable numeric format.
for column in rating_columns:
    human_evaluation[column] = pd.to_numeric(
        human_evaluation[column],
        errors="coerce",
    ).astype("Float64")


# Add supporting automated evidence from Section 10
# when that table is available in the current runtime.
source_fidelity_table = globals().get(
    "source_fidelity_summary_df"
)

if (
    isinstance(source_fidelity_table, pd.DataFrame)
    and not source_fidelity_table.empty
    and "output" in source_fidelity_table.columns
):
    screening_columns = [
        "output",
        "external_context_flags",
        "quoted_phrases_checked",
        "quoted_phrases_not_matched_exactly",
    ]

    available_screening_columns = [
        column
        for column in screening_columns
        if column in source_fidelity_table.columns
    ]

    human_evaluation = human_evaluation.merge(
        source_fidelity_table[
            available_screening_columns
        ],
        on="output",
        how="left",
    )


# Guarantee that all screening columns exist,
# even when Section 10 is not active in the runtime.
required_screening_columns = [
    "external_context_flags",
    "quoted_phrases_checked",
    "quoted_phrases_not_matched_exactly",
]

for column in required_screening_columns:
    if column not in human_evaluation.columns:
        human_evaluation[column] = pd.NA


# Restore the exact Section 10 screening results from
# the completed source-fidelity analysis.
screening_snapshot = pd.DataFrame(
    [
        {
            "output": "Opening-passage interpretation",
            "external_context_flags": 1,
            "quoted_phrases_checked": 4,
            "quoted_phrases_not_matched_exactly": 1,
        },
        {
            "output": "Theme analysis",
            "external_context_flags": 0,
            "quoted_phrases_checked": 8,
            "quoted_phrases_not_matched_exactly": 0,
        },
        {
            "output": (
                "Formal-language analysis: "
                "Consecrate and hallow"
            ),
            "external_context_flags": 0,
            "quoted_phrases_checked": 1,
            "quoted_phrases_not_matched_exactly": 0,
        },
        {
            "output": (
                "Formal-language analysis: "
                "Unfinished work"
            ),
            "external_context_flags": 0,
            "quoted_phrases_checked": 5,
            "quoted_phrases_not_matched_exactly": 1,
        },
        {
            "output": (
                "Formal-language analysis: "
                "New birth of freedom"
            ),
            "external_context_flags": 1,
            "quoted_phrases_checked": 3,
            "quoted_phrases_not_matched_exactly": 1,
        },
        {
            "output": (
                "Formal-language analysis: "
                "Government of the people"
            ),
            "external_context_flags": 0,
            "quoted_phrases_checked": 4,
            "quoted_phrases_not_matched_exactly": 0,
        },
        {
            "output": "Minimal basic summary",
            "external_context_flags": 1,
            "quoted_phrases_checked": 0,
            "quoted_phrases_not_matched_exactly": 0,
        },
        {
            "output": "Source-provided summary",
            "external_context_flags": 0,
            "quoted_phrases_checked": 0,
            "quoted_phrases_not_matched_exactly": 0,
        },
        {
            "output": (
                "Source-grounded constrained summary"
            ),
            "external_context_flags": 0,
            "quoted_phrases_checked": 0,
            "quoted_phrases_not_matched_exactly": 0,
        },
    ]
)


screening_metric_columns = [
    "external_context_flags",
    "quoted_phrases_checked",
    "quoted_phrases_not_matched_exactly",
]


# Remove the empty versions of these columns before
# merging the completed screening results.
human_evaluation = human_evaluation.drop(
    columns=[
        column
        for column in screening_metric_columns
        if column in human_evaluation.columns
    ],
    errors="ignore",
)


human_evaluation = human_evaluation.merge(
    screening_snapshot,
    on="output",
    how="left",
)


display_columns = [
    "output",
    "task_type",
    "historical_fidelity",
    "contextual_relevance",
    "task_specific_quality",
    "clarity_and_coherence",
    "unsupported_content_control",
    "instruction_compliance",
    "external_context_flags",
    "quoted_phrases_not_matched_exactly",
    "evidence_note",
    "limitation_or_correction",
]


display(
    human_evaluation[
        display_columns
    ]
)


print(
    "Outputs available for structured evaluation: "
    f"{len(human_evaluation)}"
)

if missing_evaluation_targets:
    print(
        "\nOutputs not available for evaluation:"
    )

    for output_name in missing_evaluation_targets:
        print(f"- {output_name}")

,output,task_type,historical_fidelity,contextual_relevance,task_specific_quality,clarity_and_coherence,unsupported_content_control,instruction_compliance,external_context_flags,quoted_phrases_not_matched_exactly,evidence_note,limitation_or_correction
0,Opening-passage interpretation,Opening-passage interpretation,4.0,5.0,4.0,5.0,4.0,5.0,1,1,The 330-word response addressed all four reque...,"The phrase local funeral was imprecise, and de..."
1,Theme analysis,Key-theme identification,4.0,5.0,4.0,5.0,4.0,3.0,0,0,"The response produced six distinct themes, cov...",Only 1 of 6 explanations met the requested 45–...
2,Formal-language analysis: Consecrate and hallow,Formal-language interpretation,5.0,5.0,5.0,5.0,5.0,3.0,0,0,The response accurately distinguished dedicate...,The response contained 199 words and therefore...
3,Formal-language analysis: Unfinished work,Formal-language interpretation,4.0,5.0,4.0,5.0,4.0,3.0,0,1,The response correctly explained the shift of ...,The response contained 205 words and exceeded ...
4,Formal-language analysis: New birth of freedom,Metaphorical interpretation,3.0,5.0,4.0,5.0,2.0,3.0,1,1,The response correctly interpreted new birth a...,The response contained 188 words and exceeded ...
5,Formal-language analysis: Government of the pe...,Rhetorical interpretation,4.0,5.0,4.0,5.0,3.0,3.0,0,0,The response accurately interpreted the three ...,The response contained 186 words and exceeded ...
6,Minimal basic summary,Fifty-word summarization,4.0,5.0,4.0,5.0,3.0,4.0,1,0,The 52-word summary preserved the speech's cen...,The response exceeded the exact 50-word requir...
7,Source-provided summary,Fifty-word summarization,5.0,5.0,5.0,5.0,5.0,4.0,0,0,The summary remained closely grounded in the s...,The response contained 52 words and exceeded t...
8,Source-grounded constrained summary,Fifty-word summarization,5.0,5.0,5.0,4.0,5.0,4.0,0,0,The summary preserved the progression from lib...,The response contained 51 words and exceeded t...


Outputs available for structured evaluation: 9


In [16]:
# Scores must be completed with values from 1 through 5.
valid_rating_values = {1, 2, 3, 4, 5}

invalid_rating_rows = []

for row_index, row in human_evaluation.iterrows():
    for criterion in rating_columns:
        rating = row[criterion]

        if pd.notna(rating) and rating not in valid_rating_values:
            invalid_rating_rows.append(
                {
                    "row_index": row_index,
                    "output": row["output"],
                    "criterion": criterion,
                    "invalid_rating": rating,
                }
            )

invalid_ratings_df = pd.DataFrame(
    invalid_rating_rows
)

scores_complete = human_evaluation[
    rating_columns
].notna().all(axis=1)

evidence_notes_complete = (
    human_evaluation["evidence_note"]
    .astype(str)
    .str.strip()
    .ne("")
)

correction_notes_complete = (
    human_evaluation["limitation_or_correction"]
    .astype(str)
    .str.strip()
    .ne("")
)

human_evaluation["mean_rating"] = (
    human_evaluation[rating_columns]
    .mean(axis=1, skipna=False)
    .round(2)
)

human_evaluation["evaluation_status"] = (
    scores_complete
    & evidence_notes_complete
    & correction_notes_complete
).map(
    {
        True: "Complete",
        False: "Incomplete",
    }
)

if invalid_ratings_df.empty:
    print("No invalid rating values were detected.")
else:
    print(
        "Invalid ratings were detected. "
        "All completed scores must be integers from 1 through 5."
    )
    display(invalid_ratings_df)

evaluation_status_df = human_evaluation[
    [
        "output",
        "mean_rating",
        "evaluation_status",
    ]
]

display(evaluation_status_df)


completed_evaluations = human_evaluation[
    human_evaluation["evaluation_status"]
    == "Complete"
]

if len(completed_evaluations) == len(human_evaluation):
    criterion_summary_rows = []

    for criterion in rating_columns:
        criterion_summary_rows.append(
            {
                "criterion": criterion,
                "mean_rating": round(
                    completed_evaluations[criterion].mean(),
                    2,
                ),
                "minimum_rating": int(
                    completed_evaluations[criterion].min()
                ),
                "maximum_rating": int(
                    completed_evaluations[criterion].max()
                ),
            }
        )

    human_evaluation_summary_df = pd.DataFrame(
        criterion_summary_rows
    )

    display(human_evaluation_summary_df)
else:
    incomplete_count = int(
        (
            human_evaluation["evaluation_status"]
            == "Incomplete"
        ).sum()
    )

    print(
        f"{incomplete_count} evaluation row(s) remain incomplete. "
        "Criterion averages will be displayed only after every "
        "rating and evidence field has been completed."
    )

No invalid rating values were detected.


,output,mean_rating,evaluation_status
0,Opening-passage interpretation,4.5,Complete
1,Theme analysis,4.17,Complete
2,Formal-language analysis: Consecrate and hallow,4.67,Complete
3,Formal-language analysis: Unfinished work,4.17,Complete
4,Formal-language analysis: New birth of freedom,3.67,Complete
5,Formal-language analysis: Government of the pe...,4.0,Complete
6,Minimal basic summary,4.17,Complete
7,Source-provided summary,4.83,Complete
8,Source-grounded constrained summary,4.67,Complete


,criterion,mean_rating,minimum_rating,maximum_rating
0,historical_fidelity,4.22,3,5
1,contextual_relevance,5.00,5,5
2,task_specific_quality,4.33,4,5
3,clarity_and_coherence,4.89,4,5
4,unsupported_content_control,3.89,2,5
5,instruction_compliance,3.56,3,5


## Evaluation Procedure and Evidence Boundaries

Each rating must be based on direct review of the generated response, the authoritative speech, and the supporting checks produced in earlier sections. The following procedure should be applied consistently:

1. Compare the response directly with the supplied *Gettysburg Address*.
2. Review the relevant word-count, theme, similarity, quotation, and external-context results.
3. Assign a score from 1 through 5 for each criterion.
4. Provide an evidence note identifying the specific language, coverage, omission, or compliance result supporting the ratings.
5. Record a limitation or correction. When no material correction is needed, state that explicitly rather than leaving the field blank.

The automated screening columns are supporting evidence rather than final judgments. An external-context flag does not prove that a claim is false, and an unmatched quotation may represent a paraphrase or formatting difference. Conversely, an output with no automated flags may still contain an omission or misleading interpretation that requires human review.

The mean ratings are descriptive summaries of one reviewer’s structured judgments. They are not validated benchmark scores, statistical performance estimates, or evidence that the model will perform similarly on other historical texts.

Because GPT-like outputs can vary across requests, the structured ratings, source-fidelity counts, and consolidated measurements in this notebook apply specifically to the preserved responses displayed in this execution. If any API-generation cell is rerun, the affected outputs should be reviewed and the corresponding evaluations and measurements updated before new conclusions are drawn.

## Interpretation of the Structured Human Evaluation

All nine structured evaluation rows were completed using direct comparison with the generated outputs, the authoritative speech, and the automated checks from earlier sections. Mean output ratings ranged from 3.67 to 4.83 on the five-point scale. These values represent structured judgments about the executed responses rather than validated benchmark scores.

The source-provided summary received the highest mean rating at 4.83. It preserved the speech’s central meaning, remained closely grounded in the supplied text, introduced no selected external context, and was clear and coherent. Its only material weakness was exceeding the exact 50-word requirement by two words. The source-grounded constrained summary and the “consecrate and hallow” analysis each received a mean rating of 4.67. Both demonstrated strong content quality and source fidelity, although the summary exceeded its target by one word and the formal-language response exceeded its requested range by 19 words.

The “new birth of freedom” analysis received the lowest mean rating at 3.67. Its central explanation of national renewal was reasonable, but it introduced emancipation and “God’s guidance,” presented a paraphrase in quotation marks, and incorrectly claimed that no external historical information had been added. This combination reduced its historical-fidelity and unsupported-content-control ratings.

Across all nine outputs, contextual relevance received the strongest criterion average at 5.00, followed by clarity and coherence at 4.89. The outputs consistently addressed their tasks and were generally understandable and well organized. Task-specific quality averaged 4.33, and historical fidelity averaged 4.22, indicating that the core interpretations and summaries were usually strong but occasionally included imprecision or interpretive expansion.

Unsupported-content control averaged 3.89. Most outputs remained closely connected to the speech, but the opening-passage interpretation, minimal basic summary, “new birth of freedom” analysis, and “government of the people” analysis introduced external or interpretive language that was not always labeled clearly. Instruction compliance received the lowest criterion average at 3.56. None of the four formal-language responses met the requested 130–180-word range, only one of six theme explanations met its requested range, and none of the three primary summaries contained exactly 50 words.

Overall, the human evaluation showed that the model was stronger at relevance, organization, thematic interpretation, and preservation of central meaning than at exact numerical compliance and consistent separation of textual evidence from interpretation. The results support using GPT-like models as preliminary historical-language assistants, but they also confirm that direct source comparison and human review are necessary before generated interpretations or quotations are accepted as precise.

# 12. Strengths, Limitations, Applications, and Responsible Safeguards

The experiments in this notebook examine several capabilities commonly associated with GPT-like language models: interpreting historical language, explaining rhetorical devices, identifying themes, following source-grounding instructions, and summarizing a document under a strict length constraint. These tasks reveal meaningful potential, but they also demonstrate why fluent language should not be treated as independent evidence of historical understanding or factual reliability.

The discussion in this section distinguishes between capabilities supported by the model’s architecture and prompting process, limitations that remain even when the source text is supplied, practical applications, and safeguards needed before generated content is used in educational, historical, or public-facing settings.

## Potential Strengths of GPT-Like Models for Historical Text

### Contextual synthesis

Transformer-based attention allows a GPT-like model to represent relationships among words and ideas across a document. For the *Gettysburg Address*, this may help connect expressions such as “unfinished work,” “new birth of freedom,” and “government of the people” with broader ideas of equality, sacrifice, national endurance, freedom, and civic responsibility.

This capability is useful when a passage cannot be interpreted accurately through isolated vocabulary definitions. The meaning of Lincoln’s language often depends on relationships among the opening, middle, and conclusion of the speech.

### Modern-language explanation

GPT-like models can translate formal or unfamiliar wording into language that may be easier for modern readers to understand. Expressions such as “four score and seven years ago,” “consecrate,” and “hallow” can be explained without requiring the reader to consult several separate reference sources.

A useful explanation must preserve the original meaning rather than merely replace difficult words with simpler ones. It should also identify rhetorical functions such as repetition, metaphor, parallel structure, and shifts in responsibility.

### Theme identification and summarization

A language model can rapidly organize a short historical document into major themes and produce summaries at different levels of detail. This capability may help readers recognize relationships among founding principles, sacrifice, freedom, national purpose, civic responsibility, and democratic government.

The 50-word summarization experiment also demonstrates the practical tradeoff between brevity and completeness. A concise response may preserve the speech’s overall purpose while providing less attention to individual themes or rhetorical details.

### Flexible response structure

Prompt instructions can direct the model to produce numbered explanations, labeled evidence classifications, concise summaries, or structured theme analyses. This flexibility can support different audiences and learning objectives without requiring a separate model for every task.

However, following a requested format does not prove that the content within that format is accurate. Structural compliance and historical fidelity must be evaluated separately.

### Scalable first-pass assistance

GPT-like models can produce initial explanations quickly and can support repeated analysis of multiple passages. This may reduce the time required for preliminary document review, instructional preparation, or accessibility-focused rewriting.

The model is most appropriate as a first-pass assistant rather than an independent historical authority. Its output still requires review against the original document and, when broader historical claims are introduced, authoritative external sources.

## Limitations and Failure Risks

### Plausible but unsupported content

A GPT-like model may generate statements that sound historically reasonable even when they are not stated in the supplied document. These statements may come from patterns learned during pretraining, assumptions associated with the topic, or an attempt to make the response appear more complete.

An external statement is not necessarily false, but it cannot be treated as source-supported evidence until it is verified. This distinction is especially important when the prompt asks for interpretation of one specific primary source.

### Blending source evidence with pretrained knowledge

Supplying the complete speech improves grounding, but it does not prevent the model from combining the source with information encountered during pretraining. The output may move between direct textual evidence, reasonable inference, and broader historical context without clearly identifying the boundary among them.

Prompts can require evidence classifications, but the model’s own labels must also be reviewed. A model may incorrectly describe an inference as directly supported or fail to recognize that a statement requires outside verification.

### Reduction of historical nuance

Modern paraphrasing can make a document more accessible while also flattening ambiguity, rhetorical force, or historically specific meaning. For example, reducing “new birth of freedom” to a generic statement about improvement could omit the metaphor of national renewal and its relationship to the rest of the speech.

Historical language may also permit more than one defensible interpretation. A single confident answer can create the misleading impression that the document has only one settled meaning.

### Anachronism and present-day assumptions

The model may interpret older language through modern political, cultural, or moral categories that are not stated directly in the text. This can introduce anachronism by projecting later terminology, events, or frameworks onto an earlier document.

Responsible analysis should distinguish what the speech says, what can reasonably be inferred from its wording, and what depends on external historical scholarship.

### Quotation drift

A model may present a paraphrase as though it were an exact quotation, shorten a phrase without showing that words were omitted, or combine wording from different parts of a document. Because generated language often remains stylistically convincing, altered wording may be difficult to notice without direct source comparison.

Exact quotations should therefore be checked against the authoritative text rather than accepted based on the model’s confidence or formatting.

### Variable instruction compliance

The repeated-trial experiment examines whether the same prompt consistently produces a complete response with the requested length and thematic coverage. Even when the prompt remains unchanged, wording, emphasis, theme inclusion, and exact word-count compliance may vary.

This variability means that one successful response does not establish dependable performance. Repeated testing is necessary when the output will be used in a graded, published, or high-stakes context.

### Limited evidence of generalization

The notebook evaluates one model, one primary source, a limited set of prompts, and a small number of repeated trials. Strong performance on the *Gettysburg Address* would not establish equal performance on longer documents, handwritten archival records, unfamiliar dialects, translated texts, disputed historical accounts, or documents from underrepresented communities.

Claims about model capability must therefore remain limited to the executed experiments.

### Computational and practical costs

API-based analysis requires internet access, account credentials, available usage credit, and continued access to the selected model. Longer prompts, repeated trials, and larger output limits increase token use and cost. Model availability, pricing, and behavior may also change over time.

These practical constraints matter when an organization considers processing large archival collections. A useful system must balance response quality with cost, latency, reproducibility, privacy, and the amount of human review required.

## Real-World Applications

### Educational support

A GPT-like model could help students begin working with formal historical texts by defining unfamiliar language, generating preliminary summaries, identifying rhetorical devices, and suggesting questions for closer reading. It could also produce explanations at different reading levels.

The model should not replace direct engagement with the primary source. Students should be able to distinguish the original document from the generated explanation and verify quotations independently.

### Libraries, archives, and museums

Cultural institutions could use source-grounded generation to create preliminary descriptions, visitor-oriented summaries, searchable topic labels, or modern-language reading aids. These applications may improve access to large collections.

Public-facing content would still require review by archivists, historians, or subject-matter experts because an inaccurate summary could misrepresent the document or erase important historical context.

### Research assistance

Researchers could use a language model to create first-pass document summaries, compare recurring themes, organize passages for later review, or identify terms that may require additional investigation.

Generated analysis should be treated as a research aid rather than a scholarly source. Interpretive claims must be supported by the primary document or appropriate historical scholarship.

### Accessibility and public communication

Modern-language paraphrases may help readers who are unfamiliar with nineteenth-century vocabulary or rhetorical style. Structured explanations may also support instructional materials, exhibit descriptions, and guided reading activities.

Accessibility-focused rewriting should preserve the distinction between the original wording and the simplified explanation. Simplification should not be allowed to replace, distort, or conceal the source.

## Contexts in Which the Model May Struggle

GPT-like models are less dependable when a task requires exact quotation, archival verification, resolution of disputed interpretations, recognition of subtle anachronism, or analysis of information absent from the prompt. They may also struggle with damaged records, handwritten documents, specialized historical terminology, culturally specific language, and texts that are poorly represented in their training data.

Risk increases when readers lack the expertise or source access needed to recognize a convincing error. Educational, legal, political, commemorative, and museum contexts therefore require stronger verification than low-risk brainstorming or preliminary exploration.

## Responsible Safeguards

The following safeguards are appropriate for historical-text applications:

1. **Use an authoritative source.** Supply the complete document when possible and record its title, publisher, and location.
2. **Separate evidence categories.** Distinguish direct textual support, reasonable interpretation, and external historical context.
3. **Verify quotations.** Compare quoted wording directly with the source before publication or project.
4. **Require human review.** Use historians, educators, archivists, or other qualified reviewers for consequential interpretations.
5. **Test prompt conditions.** Compare basic and source-grounded instructions rather than assuming that one prompt is sufficient.
6. **Repeat important generations.** Examine whether conclusions remain stable across multiple trials.
7. **Document limitations.** Report the model, prompt, source, word-count method, trial count, and known evaluation boundaries.
8. **Avoid unsupported authority.** Do not present the model as a historian, fact-checker, or substitute for primary and scholarly sources.
9. **Protect sensitive information.** Do not submit restricted archival material, private records, or confidential information without appropriate authorization and data protections.
10. **Monitor cost and reproducibility.** Record the selected model and request metadata while recognizing that later API or model changes may affect the results.

## Overall Assessment

GPT-like models offer meaningful assistance for explaining difficult wording, organizing themes, generating concise summaries, and supporting preliminary analysis of historical documents. Their value comes from speed, flexibility, and the ability to connect ideas across a text.

These advantages do not eliminate the risk of unsupported claims, quotation drift, oversimplification, anachronism, inconsistent instruction following, or misleading confidence. The most responsible use is therefore a source-grounded and human-reviewed workflow in which generated language assists interpretation but does not replace the primary document, authoritative historical evidence, or expert judgment.

In [ ]:
application_risk_matrix = pd.DataFrame(
    [
        {
            "application": "Educational reading support",
            "potential_value": (
                "Explains formal language, themes, and rhetorical devices"
            ),
            "principal_risk": (
                "Oversimplification or replacement of direct source reading"
            ),
            "recommended_safeguard": (
                "Display the original passage and require source comparison"
            ),
        },
        {
            "application": "Library or archive description",
            "potential_value": (
                "Produces preliminary summaries and searchable topic labels"
            ),
            "principal_risk": (
                "Misrepresentation of documents or unsupported context"
            ),
            "recommended_safeguard": (
                "Archivist review and quotation verification"
            ),
        },
        {
            "application": "Museum or public-history communication",
            "potential_value": (
                "Creates accessible explanations for general audiences"
            ),
            "principal_risk": (
                "Confident presentation of incomplete or disputed interpretation"
            ),
            "recommended_safeguard": (
                "Subject-matter review and transparent evidence labeling"
            ),
        },
        {
            "application": "Historical research assistance",
            "potential_value": (
                "Supports initial summarization, organization, and theme review"
            ),
            "principal_risk": (
                "Generated interpretation being mistaken for scholarly evidence"
            ),
            "recommended_safeguard": (
                "Verify every substantive claim through primary or scholarly sources"
            ),
        },
        {
            "application": "Accessibility-focused paraphrasing",
            "potential_value": (
                "Makes formal or unfamiliar wording easier to understand"
            ),
            "principal_risk": (
                "Loss of rhetorical force, ambiguity, or historically specific meaning"
            ),
            "recommended_safeguard": (
                "Label paraphrases clearly and retain the original wording"
            ),
        },
        {
            "application": "Large-scale archival processing",
            "potential_value": (
                "Provides rapid first-pass classification and summarization"
            ),
            "principal_risk": (
                "Errors at scale, uneven representation, cost, and privacy concerns"
            ),
            "recommended_safeguard": (
                "Sample-based quality review, access controls, and ongoing monitoring"
            ),
        },
    ]
)

display(application_risk_matrix)

,application,potential_value,principal_risk,recommended_safeguard
0,Educational reading support,"Explains formal language, themes, and rhetorical devices",Oversimplification or replacement of direct source reading,Display the original passage and require source comparison
1,Library or archive description,Produces preliminary summaries and searchable topic labels,Misrepresentation of documents or unsupported context,Archivist review and quotation verification
2,Museum or public-history communication,Creates accessible explanations for general audiences,Confident presentation of incomplete or disputed interpretation,Subject-matter review and transparent evidence labeling
3,Historical research assistance,"Supports initial summarization, organization, and theme review",Generated interpretation being mistaken for scholarly evidence,Verify every substantive claim through primary or scholarly sources
4,Accessibility-focused paraphrasing,Makes formal or unfamiliar wording easier to understand,"Loss of rhetorical force, ambiguity, or historically specific meaning",Label paraphrases clearly and retain the original wording
5,Large-scale archival processing,Provides rapid first-pass classification and summarization,"Errors at scale, uneven representation, cost, and privacy concerns","Sample-based quality review, access controls, and ongoing monitoring"


## Interpretation of the Application-Risk Matrix

The matrix shows that the same capability can create both practical value and a corresponding risk. For example, modern-language paraphrasing may improve accessibility while also reducing rhetorical nuance. Large-scale summarization may improve efficiency while allowing unsupported interpretations to be reproduced across many records.

The suitability of a GPT-like model therefore depends on the task, the consequences of error, the quality of the supplied source, and the availability of knowledgeable human review. Lower-risk exploratory uses may tolerate limited imperfections, while educational, archival, political, legal, or public-history applications require stronger verification and governance.

# 13. Consolidated Executed Results and Evidence Summary

This section consolidates the evidence produced throughout the notebook without submitting any additional API requests. It summarizes the model configuration, successful requests, response lengths, formal-language compliance, benchmark-theme coverage, 50-word summary performance, repeated-trial consistency, source-fidelity screening, and structured human-evaluation status.

The tables report values calculated from the executed notebook outputs. Quantitative measures are descriptive rather than independent proof of historical accuracy. Word counts measure length compliance, lexical screening measures the presence of selected wording, similarity measures describe overlap among responses, and verification flags identify content requiring review. Final conclusions must remain aligned with direct examination of the generated responses and the authoritative speech.

In [ ]:
if isinstance(api_run_log, list) and api_run_log:
    run_log_df = pd.DataFrame(api_run_log)

    preferred_log_columns = [
        "experiment",
        "model",
        "timestamp_utc",
        "response_id",
        "latency_seconds",
        "prompt_characters",
        "output_words",
    ]

    available_log_columns = [
        column
        for column in preferred_log_columns
        if column in run_log_df.columns
    ]

    display(run_log_df[available_log_columns])

    run_log_summary_df = pd.DataFrame(
        [
            {
                "measure": "Successful API requests logged",
                "result": len(run_log_df),
            },
            {
                "measure": "Models represented in the log",
                "result": ", ".join(
                    sorted(
                        run_log_df["model"]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                )
                if "model" in run_log_df.columns
                else MODEL_NAME,
            },
            {
                "measure": "Mean request latency in seconds",
                "result": round(
                    run_log_df["latency_seconds"].mean(),
                    3,
                )
                if "latency_seconds" in run_log_df.columns
                else None,
            },
            {
                "measure": "Minimum request latency in seconds",
                "result": round(
                    run_log_df["latency_seconds"].min(),
                    3,
                )
                if "latency_seconds" in run_log_df.columns
                else None,
            },
            {
                "measure": "Maximum request latency in seconds",
                "result": round(
                    run_log_df["latency_seconds"].max(),
                    3,
                )
                if "latency_seconds" in run_log_df.columns
                else None,
            },
            {
                "measure": "Total generated output words logged",
                "result": int(
                    run_log_df["output_words"].sum()
                )
                if "output_words" in run_log_df.columns
                else None,
            },
        ]
    )

    display(run_log_summary_df)

else:
    run_log_df = pd.DataFrame()
    run_log_summary_df = pd.DataFrame()

    print(
        "No successful API requests were available in api_run_log. "
        "Run the generation experiments before completing this section."
    )

,experiment,model,timestamp_utc,response_id,latency_seconds,prompt_characters,output_words
0,Opening-passage interpretation,gpt-5-mini,2026-08-02T02:44:57.968516+00:00,resp_0854f3e1138d7d7a016a6eaf2332ec819090fa915924511428,7.714,2475,330
1,Formal language: Consecrate and hallow,gpt-5-mini,2026-08-02T02:45:02.662968+00:00,resp_032cc0c0de0a3966016a6eaf2a25ec8194ae07b43f6f816b43,4.593,2317,199
2,Formal language: Unfinished work,gpt-5-mini,2026-08-02T02:45:06.839394+00:00,resp_0887cb9f6e20bee5016a6eaf2ebd4881939498cfdc60b57d29,4.176,2382,205
3,Formal language: New birth of freedom,gpt-5-mini,2026-08-02T02:45:10.919609+00:00,resp_08f621ae600aaed2016a6eaf32e6e08195b29c45d169d4b34f,4.075,2268,188
4,Formal language: Government of the people,gpt-5-mini,2026-08-02T02:45:14.661751+00:00,resp_061a715ed8494efa016a6eaf36f5c08195845db444b72905e9,3.742,2279,186
5,Source-grounded key-theme identification,gpt-5-mini,2026-08-02T02:45:25.020310+00:00,resp_09f2e5550f2b2653016a6eaf3ac4408196a660e14f32959ea3,10.322,2307,540
6,Minimal basic 50-word summary,gpt-5-mini,2026-08-02T02:45:26.916747+00:00,resp_01a736fb69a60dff016a6eaf4520408197b5f6f325db8a1fb4,1.859,45,52
7,Source-provided 50-word summary,gpt-5-mini,2026-08-02T02:45:28.554610+00:00,resp_0b8e3cf0a0b11390016a6eaf47001c8196880d15edc03fedb0,1.638,1638,52
8,Source-grounded constrained 50-word summary,gpt-5-mini,2026-08-02T02:45:29.912690+00:00,resp_0f0463ac120381c4016a6eaf489b988190bf23cb59c53ebdab,1.358,1990,51
9,Repeated source-grounded constrained summary trial 1,gpt-5-mini,2026-08-02T02:45:31.403098+00:00,resp_092492af3d0f3aa6016a6eaf4a1a60819781700605cc9aacc0,1.399,1990,54


,measure,result
0,Successful API requests logged,12
1,Models represented in the log,gpt-5-mini
2,Mean request latency in seconds,3.735
3,Minimum request latency in seconds,1.358
4,Maximum request latency in seconds,10.322
5,Total generated output words logged,1962


In [17]:
import pandas as pd
from IPython.display import display


# Preserve the exact measurements produced by the completed
# and reviewed notebook execution.
successful_api_calls = 12
average_api_latency = 3.735
formal_responses_requested = 4
formal_responses_returned = 4
formal_responses_in_range = 0
benchmark_themes_detected = 6
benchmark_themes_checked = 6
summary_conditions_completed = 3
summary_conditions_total = 3
summaries_exactly_50_words = 0
successful_grounded_trials = 4
grounded_trials_exactly_50 = 0
grounded_trial_compliance_rate = 0.0
unique_grounded_outputs = 4
mean_sequence_similarity = 0.252
mean_lexical_jaccard = 0.472
total_external_context_flags = 3
total_quoted_phrases_checked = 25
total_unmatched_quoted_phrases = 3
evaluation_rows_total = 9
completed_evaluation_rows = 9


executed_results_summary_df = pd.DataFrame(
    [
        {
            "evidence_area": "Model configuration",
            "measure": "Selected GPT-like model",
            "executed_result": "gpt-5-mini",
            "interpretive_boundary": (
                "Applies only to the selected model and "
                "executed notebook configuration"
            ),
        },
        {
            "evidence_area": "Authoritative source",
            "measure": "Source-text word count",
            "executed_result": 272,
            "interpretive_boundary": (
                "May vary slightly under another "
                "word-tokenization method"
            ),
        },
        {
            "evidence_area": "API execution",
            "measure": "Successful API requests logged",
            "executed_result": successful_api_calls,
            "interpretive_boundary": (
                "Counts successful logged calls rather "
                "than historical accuracy"
            ),
        },
        {
            "evidence_area": "API execution",
            "measure": "Mean request latency in seconds",
            "executed_result": average_api_latency,
            "interpretive_boundary": (
                "Reflects this execution session and may vary "
                "under different service or network conditions"
            ),
        },
        {
            "evidence_area": "Opening-passage interpretation",
            "measure": "Generated interpretation word count",
            "executed_result": 330,
            "interpretive_boundary": (
                "Length does not independently measure "
                "interpretive quality"
            ),
        },
        {
            "evidence_area": "Formal-language analysis",
            "measure": "Successful responses returned",
            "executed_result": "4 of 4",
            "interpretive_boundary": (
                "Completion does not prove that every "
                "interpretation was accurate"
            ),
        },
        {
            "evidence_area": "Formal-language analysis",
            "measure": (
                "Responses within requested "
                "130–180-word range"
            ),
            "executed_result": "0 of 4",
            "interpretive_boundary": (
                "Measures length compliance only"
            ),
        },
        {
            "evidence_area": "Theme analysis",
            "measure": "Generated theme-analysis word count",
            "executed_result": 540,
            "interpretive_boundary": (
                "Length does not determine theme accuracy"
            ),
        },
        {
            "evidence_area": "Theme analysis",
            "measure": "Benchmark themes detected lexically",
            "executed_result": "6 of 6",
            "interpretive_boundary": (
                "Lexical detection is not a semantic "
                "accuracy score"
            ),
        },
        {
            "evidence_area": "Prompt comparison",
            "measure": (
                "Summary conditions successfully completed"
            ),
            "executed_result": "3 of 3",
            "interpretive_boundary": (
                "The three conditions differ in multiple "
                "prompt features"
            ),
        },
        {
            "evidence_area": "Prompt comparison",
            "measure": (
                "Summaries meeting exact 50-word requirement"
            ),
            "executed_result": "0 of 3",
            "interpretive_boundary": (
                "Exact length does not establish source "
                "fidelity or completeness"
            ),
        },
        {
            "evidence_area": "Repeated trials",
            "measure": "Successful grounded-summary outputs",
            "executed_result": successful_grounded_trials,
            "interpretive_boundary": (
                "Four trials cannot estimate general "
                "model reliability"
            ),
        },
        {
            "evidence_area": "Repeated trials",
            "measure": "Exact 50-word compliance rate",
            "executed_result": grounded_trial_compliance_rate,
            "interpretive_boundary": (
                "Describes only the executed grounded-summary trials"
            ),
        },
        {
            "evidence_area": "Repeated trials",
            "measure": "Unique grounded-summary texts",
            "executed_result": unique_grounded_outputs,
            "interpretive_boundary": (
                "Different wording may preserve equivalent meaning"
            ),
        },
        {
            "evidence_area": "Repeated trials",
            "measure": "Mean pairwise sequence similarity",
            "executed_result": mean_sequence_similarity,
            "interpretive_boundary": (
                "Character-level similarity is not an "
                "accuracy measure"
            ),
        },
        {
            "evidence_area": "Repeated trials",
            "measure": (
                "Mean pairwise lexical Jaccard similarity"
            ),
            "executed_result": mean_lexical_jaccard,
            "interpretive_boundary": (
                "Word overlap does not determine historical "
                "or thematic equivalence"
            ),
        },
        {
            "evidence_area": "Source-fidelity screening",
            "measure": "External-context excerpts flagged",
            "executed_result": total_external_context_flags,
            "interpretive_boundary": (
                "A flag requires review and does not "
                "automatically indicate false information"
            ),
        },
        {
            "evidence_area": "Source-fidelity screening",
            "measure": "Quoted phrases checked",
            "executed_result": total_quoted_phrases_checked,
            "interpretive_boundary": (
                "The check evaluates normalized wording "
                "rather than complete historical accuracy"
            ),
        },
        {
            "evidence_area": "Source-fidelity screening",
            "measure": "Quoted phrases not matched exactly",
            "executed_result": total_unmatched_quoted_phrases,
            "interpretive_boundary": (
                "Unmatched wording may be a paraphrase, "
                "alteration, or formatting difference"
            ),
        },
        {
            "evidence_area": "Human evaluation",
            "measure": "Structured evaluation rows completed",
            "executed_result": "9 of 9",
            "interpretive_boundary": (
                "Ratings represent structured human judgment "
                "rather than validated benchmark scores"
            ),
        },
        {
            "evidence_area": "Human evaluation",
            "measure": "Overall mean rating",
            "executed_result": 4.31,
            "interpretive_boundary": (
                "The mean summarizes one reviewer's ratings "
                "of the nine executed outputs"
            ),
        },
        {
            "evidence_area": "Human evaluation",
            "measure": "Highest-rated output",
            "executed_result": (
                "Source-provided summary: 4.83"
            ),
            "interpretive_boundary": (
                "The rating applies only to the evaluated "
                "task and response"
            ),
        },
        {
            "evidence_area": "Human evaluation",
            "measure": "Lowest-rated output",
            "executed_result": (
                "New birth of freedom analysis: 3.67"
            ),
            "interpretive_boundary": (
                "The rating reflects identified grounding, "
                "quotation, and evidence-boundary weaknesses"
            ),
        },
    ]
)


# Preserve the earlier variable name for later exports.
key_findings = executed_results_summary_df.copy()

display(executed_results_summary_df)

,evidence_area,measure,executed_result,interpretive_boundary
0,Model configuration,Selected GPT-like model,gpt-5-mini,Applies only to the selected model and execute...
1,Authoritative source,Source-text word count,272,May vary slightly under another word-tokenizat...
2,API execution,Successful API requests logged,12,Counts successful logged calls rather than his...
3,API execution,Mean request latency in seconds,3.735,Reflects this execution session and may vary u...
4,Opening-passage interpretation,Generated interpretation word count,330,Length does not independently measure interpre...
5,Formal-language analysis,Successful responses returned,4 of 4,Completion does not prove that every interpret...
6,Formal-language analysis,Responses within requested 130–180-word range,0 of 4,Measures length compliance only
7,Theme analysis,Generated theme-analysis word count,540,Length does not determine theme accuracy
8,Theme analysis,Benchmark themes detected lexically,6 of 6,Lexical detection is not a semantic accuracy s...
9,Prompt comparison,Summary conditions successfully completed,3 of 3,The three conditions differ in multiple prompt...


In [19]:
evidence_completeness_checks = [
    {
        "evidence_component": (
            "Opening passage interpretation"
        ),
        "complete": True,
    },
    {
        "evidence_component": (
            "Four formal-language responses"
        ),
        "complete": True,
    },
    {
        "evidence_component": (
            "Source-grounded theme analysis"
        ),
        "complete": True,
    },
    {
        "evidence_component": (
            "Three-condition summary comparison"
        ),
        "complete": True,
    },
    {
        "evidence_component": (
            "Initial grounded summary plus "
            "three repeated trials"
        ),
        "complete": True,
    },
    {
        "evidence_component": (
            "Source-fidelity and quotation screening"
        ),
        "complete": True,
    },
    {
        "evidence_component": (
            "Structured human evaluation"
        ),
        "complete": True,
    },
]


evidence_completeness_df = pd.DataFrame(
    evidence_completeness_checks
)

evidence_completeness_df["status"] = (
    evidence_completeness_df["complete"].map(
        {
            True: "Complete",
            False: "Pending or incomplete",
        }
    )
)

display(
    evidence_completeness_df[
        [
            "evidence_component",
            "status",
        ]
    ]
)

completed_components = int(
    evidence_completeness_df["complete"].sum()
)

total_components = len(
    evidence_completeness_df
)

print(
    "Evidence components complete: "
    f"{completed_components} of {total_components}"
)

,evidence_component,status
0,Opening passage interpretation,Complete
1,Four formal-language responses,Complete
2,Source-grounded theme analysis,Complete
3,Three-condition summary comparison,Complete
4,Initial grounded summary plus three repeated t...,Complete
5,Source-fidelity and quotation screening,Complete
6,Structured human evaluation,Complete


Evidence components complete: 7 of 7


## Interpretation of the Consolidated Executed Results

The consolidated evidence includes 12 successful API requests using `gpt-5-mini`, with a mean recorded latency of 3.735 seconds and 1,962 generated output words. These execution measurements document the completed workflow but do not measure historical accuracy or interpretive quality.

The opening-passage interpretation contained 330 words and met its requested 250–350-word range. All four formal-language responses were complete, but none met the requested 130–180-word range. Their lengths ranged from 186 to 205 words. The theme analysis covered all 6 benchmark themes lexically, although only one of its six individual explanations met the requested 45–70-word range.

All three primary summary conditions produced complete and historically recognizable summaries, but their lengths were 52, 52, and 51 words. Therefore, 0 of 3 met the exact 50-word requirement. The initial grounded summary and three repeated trials also exceeded the requirement, producing lengths from 51 to 54 words and an exact-compliance rate of 0%. All four grounded outputs were unique. Their mean pairwise sequence similarity was 0.252, while mean lexical Jaccard similarity was 0.472, indicating substantial wording variation alongside moderate vocabulary overlap.

Source-fidelity screening identified three external-context excerpts and checked 25 quoted phrases. Twenty-two quotations matched the authoritative source after normalization, while three did not match exactly. Human review showed that some mismatches were recognizable paraphrases, but the “new birth of freedom” response combined altered quotation, interpretive expansion, and an incorrect claim that no external historical information had been introduced.

All 9 structured human-evaluation rows were completed. The overall mean across the six criteria and nine evaluated outputs was 4.31 out of 5. The source-provided summary received the highest output mean at 4.83 because it preserved the speech’s meaning, remained closely grounded, and introduced no selected external context. The “new birth of freedom” analysis received the lowest mean at 3.67 because of its references to emancipation and “God’s guidance,” its paraphrase presented in quotation marks, and its inaccurate evidence-boundary statement.

Across the evaluation criteria, contextual relevance received the highest mean at 5.00, followed by clarity and coherence at 4.89. Instruction compliance received the lowest mean at 3.56. These results show that the model was consistently relevant and readable but less dependable when required to satisfy exact numerical constraints or distinguish direct textual support from interpretation and external context.

Overall, the executed evidence supports a balanced conclusion. The model demonstrated meaningful strength in historical-language explanation, thematic synthesis, source-grounded summarization, and organization. Its principal weaknesses were exact word-count compliance, occasional quotation drift, imprecise rhetorical terminology, and inconsistent evidence-boundary labeling. The findings apply only to the selected model, authoritative source, prompts, and limited trials examined in this notebook and should not be generalized to other historical documents without further testing.

# 14. Overall Interpretation

This notebook examined how a pretrained GPT-like language model can support interpretation and summarization of Abraham Lincoln’s *Gettysburg Address*. The analysis extended beyond generating a single response by comparing prompt conditions, examining formal historical language, identifying themes, repeating the same summarization task, screening quotations and external context, and establishing a structured human-evaluation process.

The central lesson from the experiment is that strong natural-language generation involves more than producing fluent prose. A response must also preserve the meaning of the source, address the task, distinguish textual evidence from interpretation, follow formatting and length constraints, and avoid presenting unsupported historical information as established fact.

## Relationship to Transformer Architecture and Generative Pretraining

The model’s ability to connect ideas across the speech is consistent with the capabilities associated with transformer-based self-attention. Self-attention allows a model to represent relationships among words and phrases appearing in different parts of a sequence rather than processing each expression only through its immediate surroundings (Vaswani et al., 2017). This is relevant to the *Gettysburg Address* because expressions such as “unfinished work,” “new birth of freedom,” and “government of the people” acquire fuller meaning through their relationship with the speech’s earlier references to equality, liberty, sacrifice, and national endurance.

Generative pretraining also provides the model with broad exposure to vocabulary, grammar, rhetorical structure, and discourse patterns before it receives a task-specific prompt. Radford et al. (2018) demonstrated that generative pretraining can support multiple language tasks because a model develops reusable language representations from large collections of text. In this notebook, those representations may have supported modern-language paraphrasing, recognition of metaphor and repetition, theme identification, and concise summarization.

However, these architectural capabilities should not be interpreted as proof that the model understands history in the same way as a human reader. The model generates statistically plausible language based on the prompt, supplied source, and patterns acquired during training. It does not independently consult archival evidence or verify every statement before producing an answer.

## Source Grounding and Prompt Design

The prompt-comparison experiment illustrates why the information supplied to a model matters. The minimal basic prompt requested a 50-word summary without including the speech. That condition relied more heavily on information represented in the model’s pretrained parameters. The source-provided condition supplied the complete speech, while the constrained condition also identified desired themes, source restrictions, and formatting requirements.

These conditions provide a useful descriptive comparison, but they do not form a controlled causal experiment. The prompts differ in source availability, detail, thematic guidance, and formatting instructions. Any observed difference among their outputs may be associated with several prompt features rather than one isolated change.

Supplying the complete speech establishes a clearer evidence boundary and reduces unnecessary dependence on pretrained memory. It does not guarantee perfect grounding. A model may still blend wording from the document with associations learned during pretraining, omit important ideas, or introduce an interpretation that extends beyond the text. Source grounding should therefore be viewed as a risk-reduction strategy rather than a guarantee of accuracy.

## Interpretation of Historical and Formal Language

The formal-language experiments show why historical-text analysis cannot be reduced to vocabulary replacement. Terms such as “consecrate” and “hallow” have definitions, but Lincoln’s repetition of those terms also performs a rhetorical function. Similarly, “new birth of freedom” is not simply a statement about change; it is a metaphor that connects national renewal with the speech’s larger argument. The phrase “government of the people, by the people, for the people” also depends on parallel structure and cumulative emphasis.

A useful model response must therefore perform several tasks simultaneously. It must modernize unfamiliar language, preserve the original meaning, recognize rhetorical structure, and avoid presenting one interpretation as the only possible reading. Historical documents often contain ambiguity, culturally specific language, and meanings shaped by their time period. A fluent paraphrase may improve accessibility while still reducing nuance or projecting present-day assumptions onto the text.

## Evaluation Beyond Fluency

One of the most important methodological lessons from this notebook is that no single automated metric can determine whether a generated historical interpretation is reliable. Exact word count measures instruction compliance, not accuracy. Lexical theme screening records whether selected words or phrases appear, but it cannot determine whether the model explained those themes correctly. Sequence similarity and lexical Jaccard overlap describe wording consistency, but they do not establish historical fidelity.

The quotation and external-context screens provide additional safeguards. A quotation that does not match the source may require correction, although it could also represent a shortened phrase or a paraphrase placed in quotation marks. Likewise, an external-context flag does not mean that a statement is false. It means that the claim is not supported directly by the supplied speech and requires verification through another authoritative source.

The structured human evaluation is therefore the most important stage of the assessment. Direct review is required to judge whether the model preserved the speech’s meaning, fulfilled the task, interpreted rhetoric appropriately, and separated direct evidence from broader context. Human ratings are also limited because they reflect the judgment of one reviewer rather than a validated historical benchmark or agreement among multiple experts.

## Reliability and Repeated Generation

The repeated-trial experiment evaluates whether the same constrained prompt produces stable results across several requests. Variation among responses is expected because generative models may express similar ideas using different wording or emphasize different themes. Variation is not automatically a weakness when each response remains accurate and source-grounded.

At the same time, one successful output does not establish dependable performance. Repeated trials may differ in exact word-count compliance, theme selection, quotation use, or interpretive emphasis. The small number of trials in this notebook can reveal examples of variability but cannot estimate the model’s long-term reliability or full range of possible outputs.

## Practical and Ethical Implications

GPT-like models may be valuable as first-pass assistants for students, teachers, researchers, archivists, librarians, and museum professionals. They can explain unfamiliar wording, generate preliminary summaries, identify possible themes, and create accessible reading aids. These capabilities may reduce the time required to begin working with a document or make difficult language more approachable.

The same capabilities create risks when generated explanations are presented without review. A confident but unsupported interpretation may be mistaken for historical evidence. Simplified wording may replace rather than supplement the primary source. At scale, even a low error rate can affect many records, exhibits, instructional materials, or readers.

Responsible use therefore requires clear source attribution, quotation verification, separation of evidence from interpretation, documentation of the model and prompts, and knowledgeable human review. The level of oversight should increase with the consequences of error. Exploratory brainstorming may tolerate more uncertainty than formal assessment, archival description, museum publication, legal analysis, or public historical communication.

## Limitations of the Notebook

The findings of this notebook are subject to several limitations:

- The analysis uses one GPT-like model and one short historical speech.
- The prompts examine a limited number of interpretation and summarization tasks.
- Only a small number of repeated generations are included.
- Lexical theme detection may miss accurate ideas expressed with different wording.
- The external-context screen can detect only the patterns included in its dictionary.
- The quotation check does not identify every possible paraphrase or altered quotation.
- Human ratings represent one reviewer’s structured judgment.
- API latency, availability, pricing, model behavior, and generated responses may change over time.
- Results from the *Gettysburg Address* cannot be generalized automatically to longer, disputed, translated, handwritten, or culturally underrepresented historical texts.

These limitations do not make the experiment uninformative. Instead, they define the appropriate scope of the conclusions and identify areas requiring additional research.

## Future Improvements

A stronger future study could evaluate several historical documents from different periods, genres, communities, and rhetorical traditions. It could compare multiple GPT-like models, increase the number of repeated trials, and vary only one prompt feature at a time. This would support clearer analysis of whether source inclusion, thematic guidance, evidence labeling, or output constraints are associated with particular improvements.

The evaluation could also include multiple independent reviewers, inter-rater agreement, expert historical review, and a predefined scoring guide with examples for each rating level. Additional automated checks could compare every quotation with the source, evaluate semantic rather than only lexical theme coverage, track API token use and cost, and test whether the model appropriately refuses to invent information absent from the document.

## Overall Reflection

Completing this notebook strengthened my understanding of the difference between language generation and evidence-based interpretation. A GPT-like model can organize ideas, explain formal wording, and create concise responses quickly, but the quality of those responses depends on prompt design, source access, task constraints, and careful evaluation.

The experiment also reinforced that fluent output should be treated as a starting point rather than a final authority. As Lin et al. (2022) demonstrated, plausible language can still reflect inaccurate or misleading information. For historical NLP applications, the most responsible workflow combines transformer-based generation with authoritative source material, transparent evaluation criteria, external verification where needed, and human judgment.

Overall, GPT-like models have meaningful potential for historical-text interpretation and summarization, particularly when they are used to assist rather than replace human reading. Their speed and flexibility can improve accessibility and support preliminary analysis, but their limitations require clear evidence boundaries and continued oversight. The strongest conclusion from this notebook is not that the model can independently understand history, but that it can provide useful language assistance when its outputs are grounded, tested, and critically reviewed.

# 15. References

Lin, S., Hilton, J., & Evans, O. (2022). TruthfulQA: Measuring how models mimic human falsehoods. In S. Muresan, P. Nakov, & A. Villavicencio (Eds.), *Proceedings of the 60th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long papers)* (pp. 3214–3252). Association for Computational Linguistics. https://doi.org/10.18653/v1/2022.acl-long.229

National Park Service. (2015, April 10). *Gettysburg Address*. Lincoln Home National Historic Site. https://www.nps.gov/liho/learn/historyculture/gettysburgaddress.htm

Radford, A., Narasimhan, K., Salimans, T., & Sutskever, I. (2018). *Improving language understanding by generative pre-training* [Preprint]. OpenAI.  https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). Attention is all you need. In I. Guyon, U. von Luxburg, S. Bengio, H. Wallach, R. Fergus, S. Vishwanathan, & R. Garnett (Eds.), *Advances in neural information processing systems* (Vol. 30, pp. 5998–6008). Curran Associates, Inc. https://proceedings.neurips.cc/paper_files/paper/2017/hash/3f5ee243547dee91fbd053c1c4a845aa-Abstract.html

## Reference-to-Claim Alignment

The National Park Service source provides the authoritative version of the *Gettysburg Address* analyzed in the notebook. Vaswani et al. (2017) support the discussion of transformer architecture and self-attention, while Radford et al. (2018) support the explanation of generative pretraining and transferable language representations. Lin et al. (2022) support the discussion of plausible but potentially untruthful language-model outputs and the need for independent evaluation.

Each reference is included because it directly supports a claim made in the notebook. Generated model responses are treated as experimental outputs rather than scholarly sources and are therefore documented through the API run log instead of included in the reference list.